In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/adni_mri/adni_non_imaging")

RAW_DIR = BASE_DIR / "raw"
INTERIM_DIR = BASE_DIR / "interim"
PROCESSED_DIR = BASE_DIR / "processed"
MANIFESTS_DIR = BASE_DIR / "manifests"
QC_DIR = BASE_DIR / "qc"

for folder in [
    BASE_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    MANIFESTS_DIR,
    QC_DIR,
]:
    print(f"{folder}: {'FOUND' if folder.exists() else 'MISSING'}")

In [ ]:
import pandas as pd

manifest_files = {
    "clinical_baseline_participants": (
        MANIFESTS_DIR / "clinical_baseline_participants_with_36m_followup.csv"
    ),
    "baseline_mci_candidates": (
        MANIFESTS_DIR / "baseline_mci_candidates_with_36m_followup.csv"
    ),
    "mci_trajectory_labels": (
        MANIFESTS_DIR / "baseline_mci_36m_trajectory_labels.csv"
    ),
}

for name, path in manifest_files.items():
    print(f"\n{name}")
    print(f"Path: {path}")
    print(f"Exists: {path.exists()}")

    if path.exists():
        df_preview = pd.read_csv(path, nrows=5)
        print(f"Columns ({len(df_preview.columns)}):")
        print(df_preview.columns.tolist())
        display(df_preview)

In [ ]:
clinical_baseline = pd.read_csv(
    manifest_files["clinical_baseline_participants"]
)

baseline_mci_candidates = pd.read_csv(
    manifest_files["baseline_mci_candidates"]
)

mci_trajectory_labels = pd.read_csv(
    manifest_files["mci_trajectory_labels"]
)

datasets = {
    "clinical_baseline": clinical_baseline,
    "baseline_mci_candidates": baseline_mci_candidates,
    "mci_trajectory_labels": mci_trajectory_labels,
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns ({len(df.columns)}):")
    print(df.columns.tolist())
    display(df.head())

In [ ]:
# Standardise participant identifiers and date columns

clinical_baseline["RID"] = pd.to_numeric(
    clinical_baseline["RID_CLEAN"],
    errors="coerce"
).astype("Int64")

baseline_mci_candidates["RID"] = pd.to_numeric(
    baseline_mci_candidates["RID_CLEAN"],
    errors="coerce"
).astype("Int64")

mci_trajectory_labels["RID"] = pd.to_numeric(
    mci_trajectory_labels["RID"],
    errors="coerce"
).astype("Int64")

clinical_baseline["BASELINE_DATE"] = pd.to_datetime(
    clinical_baseline["BASELINE_DATE"],
    errors="coerce"
)

baseline_mci_candidates["BASELINE_DATE"] = pd.to_datetime(
    baseline_mci_candidates["BASELINE_DATE"],
    errors="coerce"
)

for column in [
    "BASELINE_DATE",
    "OUTCOME_WINDOW_END",
    "LAST_DIAGNOSIS_DATE",
    "FIRST_AD_DATE",
    "FIRST_CN_DATE",
    "FIRST_MCI_AT_OR_AFTER_36M",
]:
    mci_trajectory_labels[column] = pd.to_datetime(
        mci_trajectory_labels[column],
        errors="coerce"
    )

print("Clinical baseline")
print("Rows:", len(clinical_baseline))
print("Unique RID:", clinical_baseline["RID"].nunique())
print("Missing baseline dates:", clinical_baseline["BASELINE_DATE"].isna().sum())

print("\nBaseline MCI candidates")
print("Rows:", len(baseline_mci_candidates))
print("Unique RID:", baseline_mci_candidates["RID"].nunique())
print("Missing baseline dates:", baseline_mci_candidates["BASELINE_DATE"].isna().sum())

print("\nMCI trajectory labels")
print("Rows:", len(mci_trajectory_labels))
print("Unique RID:", mci_trajectory_labels["RID"].nunique())
print("Missing baseline dates:", mci_trajectory_labels["BASELINE_DATE"].isna().sum())

print("\nTrajectory label distribution:")
print(
    mci_trajectory_labels["TRAJECTORY_LABEL"]
    .value_counts(dropna=False)
)

# 1. Clinical cohort construction and baseline alignment

The two MCI-related manifests contain different numbers of participants:

- `baseline_mci_candidates_with_36m_followup.csv`: 666 participants
- `baseline_mci_36m_trajectory_labels.csv`: 1,286 participants

The trajectory-label file contains all participants identified as MCI at baseline, including participants later assigned exclusion labels because their longitudinal diagnosis history did not satisfy the final prognosis criteria.

The smaller baseline-MCI manifest appears to contain only participants marked as having sufficient diagnosis follow-up at 36 months. Before selecting the final sMCI and pMCI groups, the participant overlap between these files must be checked.

This comparison will determine:

1. whether all 666 participants are present in the trajectory-label file;
2. which trajectory outcomes are represented in the 666-person subset;
3. why 620 participants appear only in the trajectory-label file;
4. which file should be used as the authoritative source for final MCI prognosis labels.

In [ ]:
# Compare the two MCI manifest files and verify their relationship

candidate_rids = set(
    baseline_mci_candidates["RID"].dropna().astype(int)
)

trajectory_rids = set(
    mci_trajectory_labels["RID"].dropna().astype(int)
)

print("RID overlap between the two MCI files")
print("-------------------------------------")
print(f"Baseline MCI candidates: {len(candidate_rids):,}")
print(f"Trajectory-label participants: {len(trajectory_rids):,}")
print(f"Present in both: {len(candidate_rids & trajectory_rids):,}")
print(
    "Only in baseline MCI candidates:",
    len(candidate_rids - trajectory_rids)
)
print(
    "Only in trajectory labels:",
    len(trajectory_rids - candidate_rids)
)

print("\n36-month follow-up flag in baseline MCI candidates")
print("--------------------------------------------------")
print(
    baseline_mci_candidates["HAS_36M_DIAGNOSIS_FOLLOWUP"]
    .value_counts(dropna=False)
)

trajectory_membership = mci_trajectory_labels.assign(
    IN_BASELINE_MCI_CANDIDATES=
    mci_trajectory_labels["RID"].isin(candidate_rids)
)

print("\nTrajectory labels by membership in the 666-person file")
print("------------------------------------------------------")
display(
    pd.crosstab(
        trajectory_membership["TRAJECTORY_LABEL"],
        trajectory_membership["IN_BASELINE_MCI_CANDIDATES"],
        margins=True,
    )
)

### 1.1. Interpretation of the MCI manifest comparison

The comparison confirms that `baseline_mci_candidates_with_36m_followup.csv` is not the correct source for the final MCI cohort.

All 666 participants in that file are present in the trajectory-label file, but the smaller file excludes 620 additional baseline-MCI participants. Most of those additional participants have short follow-up and are correctly excluded. However, they also include:

- 125 valid pMCI participants;
- 7 valid sMCI participants.

This happens because the rules for assigning pMCI and sMCI are not identical to simply requiring the general `HAS_36M_DIAGNOSIS_FOLLOWUP` flag. For example, a participant may convert to AD within the outcome window and satisfy the pMCI trajectory rules without appearing in the narrower 36-month-follow-up manifest.

Therefore:

- `baseline_mci_36m_trajectory_labels.csv` should be used as the authoritative source for MCI outcomes;
- only rows labelled `pMCI` or `sMCI` should enter the final cohort;
- all rows with labels beginning with `exclude_` should remain excluded;
- `baseline_mci_candidates_with_36m_followup.csv` is an intermediate quality-control subset and should not be used to define the final MCI sample.

The broader `clinical_baseline_participants_with_36m_followup.csv` file also contains only 1,337 participants. This is much smaller than the previously established augmented four-group clinical cohort, so it should not yet be assumed to be the correct source for the CN and AD groups. The manifests folder must be searched for the final four-group or augmented cohort file before proceeding.

# 2. Reconstruct the authoritative clinical cohort manifest from DXSUM

The final clinical manifest will be rebuilt directly from the cleaned longitudinal diagnosis history in `DXSUM`.

This manifest will provide one authoritative baseline record per participant and will contain:

- participant identifiers;
- baseline phase and baseline diagnosis date;
- baseline diagnosis;
- longitudinal diagnosis trajectory;
- final cohort label:
  - `CN`;
  - `AD`;
  - `sMCI`;
  - `pMCI`;
- prognosis target for the MCI groups;
- conversion and follow-up dates where relevant;
- exclusion reason for baseline-MCI participants who do not satisfy the trajectory rules.

The reconstruction will use the previously established 36-month prognosis rules. It will not require participants to have MRI or any other modality.

The completed files will be saved in a dedicated folder inside `manifests` so they remain separate from older intermediate cohort files.

# 3. Load DXSUM and prepare the authoritative clinical cohort folder

The clinical cohort will be reconstructed directly from the full longitudinal DXSUM table:

`All_Subjects_DXSUM_11Jul2026.csv`

DXSUM will be used to determine:

- each participant's valid baseline diagnosis;
- the baseline diagnosis date;
- all subsequent diagnosis events;
- MCI conversion or stability over the 36-month outcome window;
- the final `CN`, `AD`, `sMCI`, or `pMCI` cohort label;
- exclusion reasons for baseline-MCI participants who do not satisfy the prognosis rules.

The cohort will not be restricted by MRI or by the availability of any other modality. The resulting files will be saved in a new authoritative clinical cohort folder inside `manifests`.

In [ ]:
DXSUM_PATH = (
    RAW_DIR
    / "Cohort, dates and source-of-truth tables"
    / "All_Subjects_DXSUM_11Jul2026.csv"
)

CLINICAL_COHORT_DIR = (
    MANIFESTS_DIR
    / "authoritative_clinical_cohort"
)

CLINICAL_COHORT_DIR.mkdir(parents=True, exist_ok=True)

print("DXSUM path:")
print(DXSUM_PATH)
print("Exists:", DXSUM_PATH.exists())

print("\nClinical cohort output folder:")
print(CLINICAL_COHORT_DIR)
print("Exists:", CLINICAL_COHORT_DIR.exists())

dxsum_raw = pd.read_csv(DXSUM_PATH, low_memory=False)

print(f"\nDXSUM rows: {len(dxsum_raw):,}")
print(f"DXSUM columns: {len(dxsum_raw.columns)}")
print("\nColumns:")
print(dxsum_raw.columns.tolist())

# 4. Standardise the core DXSUM diagnosis fields

The raw DXSUM table contains repeated longitudinal diagnosis records across ADNI phases. Before reconstructing the cohort, the key participant, visit, date, and diagnosis fields must be standardised.

This step will:

- convert `RID` to a consistent integer identifier;
- standardise `PTID`, `PHASE`, `VISCODE`, and `VISCODE2`;
- parse `EXAMDATE` as a date;
- convert `DIAGNOSIS` to a numeric code;
- map the official diagnosis codes:
  - `1 = CN`;
  - `2 = MCI`;
  - `3 = AD`;
- flag rows with missing identifiers, dates, or valid diagnosis codes;
- inspect the number of valid longitudinal diagnosis events available for cohort reconstruction.

In [ ]:
# Standardise the core DXSUM fields

dxsum = dxsum_raw.copy()

dxsum["RID"] = pd.to_numeric(
    dxsum["RID"],
    errors="coerce"
).astype("Int64")

for column in ["PTID", "PHASE", "VISCODE", "VISCODE2"]:
    dxsum[column] = (
        dxsum[column]
        .astype("string")
        .str.strip()
    )

dxsum["EXAMDATE"] = pd.to_datetime(
    dxsum["EXAMDATE"],
    errors="coerce"
)

dxsum["DIAGNOSIS"] = pd.to_numeric(
    dxsum["DIAGNOSIS"],
    errors="coerce"
).astype("Int64")

diagnosis_map = {
    1: "CN",
    2: "MCI",
    3: "AD",
}

dxsum["DIAGNOSIS_LABEL"] = dxsum["DIAGNOSIS"].map(diagnosis_map)

dxsum["HAS_VALID_RID"] = dxsum["RID"].notna()
dxsum["HAS_VALID_DATE"] = dxsum["EXAMDATE"].notna()
dxsum["HAS_VALID_DIAGNOSIS"] = dxsum["DIAGNOSIS_LABEL"].notna()

dxsum["IS_VALID_DIAGNOSIS_EVENT"] = (
    dxsum["HAS_VALID_RID"]
    & dxsum["HAS_VALID_DATE"]
    & dxsum["HAS_VALID_DIAGNOSIS"]
)

print("DXSUM standardisation summary")
print("-----------------------------")
print(f"Total rows: {len(dxsum):,}")
print(f"Unique RID: {dxsum['RID'].nunique():,}")
print(f"Rows missing RID: {dxsum['RID'].isna().sum():,}")
print(f"Rows missing EXAMDATE: {dxsum['EXAMDATE'].isna().sum():,}")
print(f"Rows with invalid or missing DIAGNOSIS: {dxsum['DIAGNOSIS_LABEL'].isna().sum():,}")
print(f"Valid dated diagnosis events: {dxsum['IS_VALID_DIAGNOSIS_EVENT'].sum():,}")

print("\nDiagnosis distribution among valid events:")
print(
    dxsum.loc[
        dxsum["IS_VALID_DIAGNOSIS_EVENT"],
        "DIAGNOSIS_LABEL"
    ].value_counts(dropna=False)
)

print("\nVISCODE2 distribution for valid diagnosis events:")
print(
    dxsum.loc[
        dxsum["IS_VALID_DIAGNOSIS_EVENT"],
        "VISCODE2"
    ].value_counts(dropna=False).head(20)
)

# 5. Audit screening and baseline diagnosis records

Before defining the authoritative baseline cohort, screening visits must be compared with formal baseline visits.

This step will identify:

- participants with a valid `bl` diagnosis;
- participants with a valid `sc` diagnosis but no valid `bl` diagnosis;
- participants who have both `sc` and `bl`;
- whether the screening and baseline diagnoses agree;
- the number of days between screening and baseline.

The formal `bl` visit will remain the preferred index visit. Screening-only participants will not be added automatically; they will first be inspected as a separate group.

In [ ]:
# Compare valid screening and baseline diagnosis records

valid_dx = dxsum.loc[
    dxsum["IS_VALID_DIAGNOSIS_EVENT"]
].copy()

screening = (
    valid_dx.loc[
        valid_dx["VISCODE2"].eq("sc"),
        ["RID", "PTID", "PHASE", "EXAMDATE", "DIAGNOSIS", "DIAGNOSIS_LABEL"]
    ]
    .sort_values(["RID", "EXAMDATE"])
    .drop_duplicates(subset="RID", keep="first")
    .rename(columns={
        "PHASE": "SC_PHASE",
        "EXAMDATE": "SC_DATE",
        "DIAGNOSIS": "SC_DIAGNOSIS_CODE",
        "DIAGNOSIS_LABEL": "SC_DIAGNOSIS",
    })
)

baseline = (
    valid_dx.loc[
        valid_dx["VISCODE2"].eq("bl"),
        ["RID", "PTID", "PHASE", "EXAMDATE", "DIAGNOSIS", "DIAGNOSIS_LABEL"]
    ]
    .sort_values(["RID", "EXAMDATE"])
    .drop_duplicates(subset="RID", keep="first")
    .rename(columns={
        "PHASE": "BL_PHASE",
        "EXAMDATE": "BL_DATE",
        "DIAGNOSIS": "BL_DIAGNOSIS_CODE",
        "DIAGNOSIS_LABEL": "BL_DIAGNOSIS",
    })
)

sc_bl_audit = screening.merge(
    baseline,
    on="RID",
    how="outer",
    suffixes=("_SC", "_BL"),
)

sc_bl_audit["HAS_SC"] = sc_bl_audit["SC_DATE"].notna()
sc_bl_audit["HAS_BL"] = sc_bl_audit["BL_DATE"].notna()

sc_bl_audit["SC_BL_DAY_GAP"] = (
    sc_bl_audit["BL_DATE"] - sc_bl_audit["SC_DATE"]
).dt.days

sc_bl_audit["SC_BL_DIAGNOSIS_AGREES"] = (
    sc_bl_audit["SC_DIAGNOSIS"].eq(sc_bl_audit["BL_DIAGNOSIS"])
)

print("Screening and baseline availability")
print("-----------------------------------")
print(f"Participants with valid screening diagnosis: {sc_bl_audit['HAS_SC'].sum():,}")
print(f"Participants with valid baseline diagnosis: {sc_bl_audit['HAS_BL'].sum():,}")
print(f"Participants with both: {(sc_bl_audit['HAS_SC'] & sc_bl_audit['HAS_BL']).sum():,}")
print(f"Screening only: {(sc_bl_audit['HAS_SC'] & ~sc_bl_audit['HAS_BL']).sum():,}")
print(f"Baseline only: {(~sc_bl_audit['HAS_SC'] & sc_bl_audit['HAS_BL']).sum():,}")

both_mask = sc_bl_audit["HAS_SC"] & sc_bl_audit["HAS_BL"]

print("\nDiagnosis agreement where both visits exist")
print("-------------------------------------------")
print(
    sc_bl_audit.loc[
        both_mask,
        "SC_BL_DIAGNOSIS_AGREES"
    ].value_counts(dropna=False)
)

print("\nScreening-to-baseline day-gap summary")
print("-------------------------------------")
print(
    sc_bl_audit.loc[
        both_mask,
        "SC_BL_DAY_GAP"
    ].describe()
)

print("\nDiagnosis distribution among screening-only participants")
print("--------------------------------------------------------")
print(
    sc_bl_audit.loc[
        sc_bl_audit["HAS_SC"] & ~sc_bl_audit["HAS_BL"],
        "SC_DIAGNOSIS"
    ].value_counts(dropna=False)
)

# 6. Audit whether screening-only participants continued into follow-up

Participants with a valid screening diagnosis but no formal baseline diagnosis should not be excluded immediately.

Some may have:

- continued into ADNI and completed later diagnosis visits;
- had their baseline visit recorded under a non-standard visit code;
- had no formal `bl` diagnosis despite having other baseline-era assessments;
- genuinely failed to proceed beyond screening.

This step will examine all later valid diagnosis events for screening-only participants and determine:

- how many have any diagnosis after screening;
- the first later diagnosis visit code and date;
- the delay from screening to the first later diagnosis;
- whether the later diagnosis agrees with the screening diagnosis;
- how many appear to have continued in the study despite lacking a formal `bl` row.

These participants will remain separate until a defensible fallback baseline rule is established.

In [ ]:
# Inspect longitudinal diagnosis follow-up for screening-only participants

screening_only_rids = set(
    sc_bl_audit.loc[
        sc_bl_audit["HAS_SC"] & ~sc_bl_audit["HAS_BL"],
        "RID"
    ].dropna().astype(int)
)

screening_only_events = valid_dx.loc[
    valid_dx["RID"].isin(screening_only_rids)
].copy()

screening_only_events = screening_only_events.sort_values(
    ["RID", "EXAMDATE", "VISCODE2"]
)

screening_only_summary = (
    screening_only_events
    .groupby("RID", as_index=False)
    .agg(
        PTID=("PTID", "first"),
        PHASE=("PHASE", "first"),
        FIRST_EVENT_DATE=("EXAMDATE", "min"),
        LAST_EVENT_DATE=("EXAMDATE", "max"),
        VALID_DIAGNOSIS_EVENT_COUNT=("EXAMDATE", "size"),
        UNIQUE_VISCODE2_COUNT=("VISCODE2", "nunique"),
    )
)

screening_reference = (
    screening_only_events.loc[
        screening_only_events["VISCODE2"].eq("sc"),
        ["RID", "EXAMDATE", "DIAGNOSIS_LABEL"]
    ]
    .sort_values(["RID", "EXAMDATE"])
    .drop_duplicates("RID", keep="first")
    .rename(columns={
        "EXAMDATE": "SC_DATE",
        "DIAGNOSIS_LABEL": "SC_DIAGNOSIS",
    })
)

later_events = screening_only_events.loc[
    ~screening_only_events["VISCODE2"].eq("sc")
].copy()

first_later_event = (
    later_events
    .sort_values(["RID", "EXAMDATE"])
    .drop_duplicates("RID", keep="first")
    [["RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS_LABEL"]]
    .rename(columns={
        "VISCODE2": "FIRST_LATER_VISCODE2",
        "EXAMDATE": "FIRST_LATER_DATE",
        "DIAGNOSIS_LABEL": "FIRST_LATER_DIAGNOSIS",
    })
)

screening_only_audit = (
    screening_only_summary
    .merge(screening_reference, on="RID", how="left")
    .merge(first_later_event, on="RID", how="left")
)

screening_only_audit["HAS_LATER_DIAGNOSIS"] = (
    screening_only_audit["FIRST_LATER_DATE"].notna()
)

screening_only_audit["DAYS_SC_TO_FIRST_LATER"] = (
    screening_only_audit["FIRST_LATER_DATE"]
    - screening_only_audit["SC_DATE"]
).dt.days

screening_only_audit["SC_LATER_DIAGNOSIS_AGREES"] = (
    screening_only_audit["SC_DIAGNOSIS"]
    .eq(screening_only_audit["FIRST_LATER_DIAGNOSIS"])
)

print("Screening-only continuation audit")
print("---------------------------------")
print(f"Screening-only participants: {len(screening_only_audit):,}")
print(
    "With any later valid diagnosis event:",
    f"{screening_only_audit['HAS_LATER_DIAGNOSIS'].sum():,}"
)
print(
    "With no later valid diagnosis event:",
    f"{(~screening_only_audit['HAS_LATER_DIAGNOSIS']).sum():,}"
)

print("\nFirst later visit-code distribution:")
print(
    screening_only_audit["FIRST_LATER_VISCODE2"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nScreening-to-first-later-event day-gap summary:")
print(
    screening_only_audit.loc[
        screening_only_audit["HAS_LATER_DIAGNOSIS"],
        "DAYS_SC_TO_FIRST_LATER"
    ].describe()
)

print("\nScreening versus first later diagnosis agreement:")
print(
    screening_only_audit.loc[
        screening_only_audit["HAS_LATER_DIAGNOSIS"],
        "SC_LATER_DIAGNOSIS_AGREES"
    ].value_counts(dropna=False)
)

### 6.1. Screening-only participant decision

Participants with a valid screening diagnosis but no formal baseline diagnosis will be excluded.

Of 741 screening-only participants, 736 had no later valid diagnosis record, while the remaining 5 reappeared only after long gaps of 387-1,148 days. These records do not provide a reliable baseline for cohort construction.

# 7. Construct the formal baseline diagnosis table

Only participants with a valid `VISCODE2 = "bl"` diagnosis will be retained for cohort reconstruction.

Before assigning cohort labels, this step will verify that each participant has:

- one valid baseline diagnosis date;
- one baseline diagnosis code;
- no conflicting diagnosis records on the same baseline date.

Participants with duplicate but identical baseline records may be safely collapsed. Participants with conflicting baseline diagnoses will be flagged for inspection.

In [ ]:
# Build and audit the formal baseline diagnosis table

baseline_events = (
    valid_dx.loc[
        valid_dx["VISCODE2"].eq("bl"),
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
            "DIAGNOSIS",
            "DIAGNOSIS_LABEL",
        ],
    ]
    .copy()
    .sort_values(["RID", "EXAMDATE", "DIAGNOSIS"])
)

baseline_audit = (
    baseline_events
    .groupby("RID", as_index=False)
    .agg(
        BASELINE_ROW_COUNT=("RID", "size"),
        BASELINE_DATE_COUNT=("EXAMDATE", "nunique"),
        BASELINE_DIAGNOSIS_COUNT=("DIAGNOSIS_LABEL", "nunique"),
    )
)

baseline_audit["HAS_DUPLICATE_BASELINE_ROWS"] = (
    baseline_audit["BASELINE_ROW_COUNT"] > 1
)

baseline_audit["HAS_MULTIPLE_BASELINE_DATES"] = (
    baseline_audit["BASELINE_DATE_COUNT"] > 1
)

baseline_audit["HAS_BASELINE_DIAGNOSIS_CONFLICT"] = (
    baseline_audit["BASELINE_DIAGNOSIS_COUNT"] > 1
)

print("Formal baseline audit")
print("---------------------")
print(f"Participants with valid baseline diagnosis: {len(baseline_audit):,}")
print(
    "Participants with duplicate baseline rows:",
    f"{baseline_audit['HAS_DUPLICATE_BASELINE_ROWS'].sum():,}"
)
print(
    "Participants with multiple baseline dates:",
    f"{baseline_audit['HAS_MULTIPLE_BASELINE_DATES'].sum():,}"
)
print(
    "Participants with conflicting baseline diagnoses:",
    f"{baseline_audit['HAS_BASELINE_DIAGNOSIS_CONFLICT'].sum():,}"
)

print("\nBaseline diagnosis distribution:")
print(
    baseline_events
    .drop_duplicates(subset=["RID"])
    ["DIAGNOSIS_LABEL"]
    .value_counts(dropna=False)
)

### 7.1. Formal baseline audit summary

The formal baseline cohort contains 2,944 participants:

- 1,196 CN;
- 1,286 MCI;
- 462 AD.

Each participant has exactly one valid baseline diagnosis row, with no duplicate baseline dates and no conflicting baseline diagnoses. This table can therefore be used as the authoritative starting point for cohort reconstruction.

# 8. Verify DXSUM diagnosis coding using the ADNI data dictionary

Before assigning clinical labels, the `DIAGNOSIS` field in DXSUM will be verified against the official ADNI data dictionary.

The dictionary file `DATADIC_11Jul2026.csv` will be loaded from the same source-of-truth folder and filtered specifically for the `DIAGNOSIS` field associated with DXSUM. All matching entries will be displayed without assuming the table name, phase coverage, or diagnosis-code mapping in advance.

In [ ]:
# Load DATADIC and locate the official definition of the DXSUM diagnosis field

DATADIC_PATH = (
    RAW_DIR
    / "Cohort, dates and source-of-truth tables"
    / "DATADIC_11Jul2026.csv"
)

print("DATADIC path:")
print(DATADIC_PATH)
print("Exists:", DATADIC_PATH.exists())

datadic = pd.read_csv(DATADIC_PATH, low_memory=False)

# Standardise relevant dictionary fields only for searching
for column in ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME"]:
    datadic[column] = (
        datadic[column]
        .astype("string")
        .str.strip()
    )

diagnosis_dictionary_matches = datadic.loc[
    datadic["FLDNAME"].str.upper().eq("DIAGNOSIS")
    & (
        datadic["TBLNAME"].str.upper().str.contains("DXSUM", na=False)
        | datadic["CRFNAME"].str.upper().str.contains(
            "DIAGNOSIS", na=False
        )
    )
].copy()

display_columns = [
    "PHASE",
    "CRFNAME",
    "TBLNAME",
    "FLDNAME",
    "TEXT",
    "TYPE",
    "LENGTH",
    "DD_CRF_VERSION",
    "CODE",
    "UNITS",
    "STATUS",
    "CODE_CHANGES",
    "MAPPING_NOTES",
]

print(f"\nDATADIC rows: {len(datadic):,}")
print(
    "Matching diagnosis-definition rows:",
    f"{len(diagnosis_dictionary_matches):,}"
)

display(
    diagnosis_dictionary_matches[
        display_columns
    ].sort_values(
        ["TBLNAME", "PHASE", "DD_CRF_VERSION"],
        na_position="last",
    )
)

### 8.1. Diagnosis coding verification

The ADNI data dictionary confirms that the `DIAGNOSIS` field is coded as:

- `1 = CN`
- `2 = MCI`
- `3 = Dementia`

Therefore, code `3` should not yet be relabelled automatically as AD. The DXSUM table contains additional dementia-cause fields, including `DXAD`, which must be checked before identifying the AD cohort.

The next step will inspect the official dictionary definition and observed values of `DXAD`.

In [ ]:
# Inspect the official definition and observed values of DXSUM.DXAD

dxad_dictionary_matches = datadic.loc[
    datadic["FLDNAME"].str.upper().eq("DXAD")
    & datadic["TBLNAME"].str.upper().eq("DXSUM")
].copy()

print("Official DATADIC entries for DXSUM.DXAD")
print("-----------------------------------------")
print(f"Rows found: {len(dxad_dictionary_matches):,}")

display(
    dxad_dictionary_matches[
        [
            "PHASE",
            "CRFNAME",
            "TBLNAME",
            "FLDNAME",
            "TEXT",
            "TYPE",
            "LENGTH",
            "DD_CRF_VERSION",
            "CODE",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ].sort_values(
        ["PHASE", "DD_CRF_VERSION"],
        na_position="last",
    )
)

print("\nObserved DXAD values among valid diagnosis events")
print("------------------------------------------------")
print(
    pd.to_numeric(
        dxsum.loc[
            dxsum["IS_VALID_DIAGNOSIS_EVENT"],
            "DXAD"
        ],
        errors="coerce"
    ).value_counts(dropna=False)
)

print("\nDXAD values among DIAGNOSIS = 3 records")
print("---------------------------------------")
print(
    pd.to_numeric(
        dxsum.loc[
            dxsum["IS_VALID_DIAGNOSIS_EVENT"]
            & dxsum["DIAGNOSIS"].eq(3),
            "DXAD"
        ],
        errors="coerce"
    ).value_counts(dropna=False)
)

# 9. Audit the cause of baseline dementia diagnoses

Participants with `DIAGNOSIS = 3` are documented as having dementia, but not all dementia records can automatically be treated as Alzheimer’s disease.

This step will inspect the baseline dementia rows and compare the available dementia-cause fields, including:

- `DXAD`;
- `DXOTHDEM`;
- `DXPARK`;
- `DXDEP`;
- `DXODES`;
- `DXDDUE`.

The aim is to determine how many baseline dementia participants are explicitly confirmed as Alzheimer’s disease and how many require further review.

In [ ]:
# Audit dementia-cause fields among formal baseline dementia participants

baseline_dementia = dxsum.loc[
    dxsum["IS_VALID_DIAGNOSIS_EVENT"]
    & dxsum["VISCODE2"].eq("bl")
    & dxsum["DIAGNOSIS"].eq(3)
].copy()

cause_fields = [
    "DXAD",
    "DXOTHDEM",
    "DXPARK",
    "DXDEP",
    "DXODES",
    "DXDDUE",
]

for column in cause_fields:
    baseline_dementia[column] = pd.to_numeric(
        baseline_dementia[column],
        errors="coerce"
    )

print("Baseline dementia participants")
print("------------------------------")
print(f"Rows: {len(baseline_dementia):,}")
print(f"Unique RID: {baseline_dementia['RID'].nunique():,}")

print("\nObserved values in dementia-cause fields")
print("----------------------------------------")

for column in cause_fields:
    print(f"\n{column}")
    print(
        baseline_dementia[column]
        .value_counts(dropna=False)
        .sort_index()
    )

baseline_dementia["CONFIRMED_AD"] = baseline_dementia["DXAD"].eq(1)

baseline_dementia["HAS_OTHER_DEMENTIA_CAUSE"] = (
    baseline_dementia[
        ["DXOTHDEM", "DXPARK", "DXDEP", "DXODES"]
    ]
    .eq(1)
    .any(axis=1)
)

print("\nBaseline dementia classification audit")
print("--------------------------------------")
print(
    baseline_dementia[
        ["CONFIRMED_AD", "HAS_OTHER_DEMENTIA_CAUSE"]
    ]
    .value_counts(dropna=False)
)

display(
    baseline_dementia[
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
            "DIAGNOSIS",
            "DXAD",
            "DXOTHDEM",
            "DXPARK",
            "DXDEP",
            "DXODES",
            "DXDDUE",
            "CONFIRMED_AD",
            "HAS_OTHER_DEMENTIA_CAUSE",
        ]
    ].head(20)
)

# 10. Verify the official meaning of dementia-cause fields

The observed baseline dementia records show two different documentation patterns:

- 193 ADNI1 records use `DXAD = 1`;
- 269 later-phase records mainly use `DXDDUE`, while `DXAD` is missing.

Therefore, the dementia-cause fields must be interpreted using their official data-dictionary definitions before deciding which baseline dementia participants can be labelled as AD.

In [ ]:
# Inspect official DATADIC definitions for dementia-cause fields used in DXSUM

dementia_cause_fields = [
    "DXAD",
    "DXDDUE",
    "DXOTHDEM",
    "DXPARK",
    "DXDEP",
    "DXODES",
]

dementia_cause_dictionary = datadic.loc[
    datadic["TBLNAME"].str.upper().eq("DXSUM")
    & datadic["FLDNAME"].str.upper().isin(dementia_cause_fields)
].copy()

display(
    dementia_cause_dictionary[
        [
            "PHASE",
            "CRFNAME",
            "TBLNAME",
            "FLDNAME",
            "TEXT",
            "TYPE",
            "LENGTH",
            "DD_CRF_VERSION",
            "CODE",
            "STATUS",
            "CODE_CHANGES",
            "MAPPING_NOTES",
        ]
    ].sort_values(
        ["FLDNAME", "PHASE", "DD_CRF_VERSION"],
        na_position="last",
    )
)

# 11. Define phase-specific Alzheimer’s disease confirmation rules

The data dictionary shows that DXSUM records Alzheimer’s disease differently across ADNI phases:

- in `ADNI1`, Alzheimer’s disease is explicitly recorded using `DXAD = 1`;
- in `ADNIGO`, `ADNI2`, `ADNI3`, and `ADNI4`, dementia cause is recorded using `DXDDUE`, where `DXDDUE = 1` means dementia due to Alzheimer’s disease.

Fields such as `DXDEP` and `DXPARK` describe accompanying symptoms and should not by themselves be treated as alternative dementia causes.

Before assigning the final AD label, the phase-specific rule will be applied and its results checked across all baseline dementia participants.

In [ ]:
# Apply the official phase-specific Alzheimer’s disease confirmation rule

baseline_dementia["AD_CONFIRMATION_SOURCE"] = pd.NA

adni1_ad_mask = (
    baseline_dementia["PHASE"].eq("ADNI1")
    & baseline_dementia["DXAD"].eq(1)
)

later_phase_ad_mask = (
    baseline_dementia["PHASE"].isin(
        ["ADNIGO", "ADNI2", "ADNI3", "ADNI4"]
    )
    & baseline_dementia["DXDDUE"].eq(1)
)

baseline_dementia.loc[
    adni1_ad_mask,
    "AD_CONFIRMATION_SOURCE"
] = "DXAD = 1"

baseline_dementia.loc[
    later_phase_ad_mask,
    "AD_CONFIRMATION_SOURCE"
] = "DXDDUE = 1"

baseline_dementia["CONFIRMED_AD_OFFICIAL"] = (
    baseline_dementia["AD_CONFIRMATION_SOURCE"].notna()
)

print("Official baseline AD confirmation")
print("---------------------------------")
print(
    baseline_dementia["CONFIRMED_AD_OFFICIAL"]
    .value_counts(dropna=False)
)

print("\nConfirmed AD by phase")
print("---------------------")
display(
    pd.crosstab(
        baseline_dementia["PHASE"],
        baseline_dementia["CONFIRMED_AD_OFFICIAL"],
        margins=True,
    )
)

print("\nConfirmation source by phase")
print("----------------------------")
display(
    pd.crosstab(
        baseline_dementia["PHASE"],
        baseline_dementia["AD_CONFIRMATION_SOURCE"],
        margins=True,
        dropna=False,
    )
)

print("\nUnconfirmed baseline dementia records")
print("-------------------------------------")
display(
    baseline_dementia.loc[
        ~baseline_dementia["CONFIRMED_AD_OFFICIAL"],
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
            "DIAGNOSIS",
            "DXAD",
            "DXDDUE",
            "DXODES",
            "DXOTHDEM",
        ],
    ].sort_values(["PHASE", "RID"])
)

### 11.1. Baseline dementia audit summary

Of the 462 participants diagnosed with dementia at baseline:

- 459 are explicitly confirmed as Alzheimer’s disease using the phase-appropriate DXSUM field;
- 3 ADNI4 participants have `DXDDUE = 2`, indicating dementia due to another etiology.

The three non-Alzheimer’s dementia participants will not be included in the AD cohort. The same phase-specific confirmation rule must now be applied to every longitudinal dementia event before reconstructing MCI conversion trajectories.

# 12. Harmonise longitudinal diagnosis labels across ADNI phases

All valid DXSUM diagnosis events will now receive a harmonised clinical label.

The documented rules are:

- `DIAGNOSIS = 1` → `CN`;
- `DIAGNOSIS = 2` → `MCI`;
- `DIAGNOSIS = 3` with `DXAD = 1` in ADNI1 → `AD`;
- `DIAGNOSIS = 3` with `DXDDUE = 1` in ADNIGO, ADNI2, ADNI3, or ADNI4 → `AD`;
- other `DIAGNOSIS = 3` records → `OTHER_DEMENTIA`.

Only confirmed `AD` events will count as conversion events when constructing pMCI trajectories.

In [ ]:
# Create phase-aware harmonised diagnosis labels for all valid dated events

longitudinal_dx = dxsum.loc[
    dxsum["IS_VALID_DIAGNOSIS_EVENT"]
].copy()

longitudinal_dx["DXAD"] = pd.to_numeric(
    longitudinal_dx["DXAD"],
    errors="coerce"
)

longitudinal_dx["DXDDUE"] = pd.to_numeric(
    longitudinal_dx["DXDDUE"],
    errors="coerce"
)

longitudinal_dx["HARMONISED_DIAGNOSIS"] = pd.NA

longitudinal_dx.loc[
    longitudinal_dx["DIAGNOSIS"].eq(1),
    "HARMONISED_DIAGNOSIS"
] = "CN"

longitudinal_dx.loc[
    longitudinal_dx["DIAGNOSIS"].eq(2),
    "HARMONISED_DIAGNOSIS"
] = "MCI"

confirmed_ad_event = (
    (
        longitudinal_dx["PHASE"].eq("ADNI1")
        & longitudinal_dx["DIAGNOSIS"].eq(3)
        & longitudinal_dx["DXAD"].eq(1)
    )
    |
    (
        longitudinal_dx["PHASE"].isin(
            ["ADNIGO", "ADNI2", "ADNI3", "ADNI4"]
        )
        & longitudinal_dx["DIAGNOSIS"].eq(3)
        & longitudinal_dx["DXDDUE"].eq(1)
    )
)

longitudinal_dx.loc[
    confirmed_ad_event,
    "HARMONISED_DIAGNOSIS"
] = "AD"

longitudinal_dx.loc[
    longitudinal_dx["DIAGNOSIS"].eq(3)
    & ~confirmed_ad_event,
    "HARMONISED_DIAGNOSIS"
] = "OTHER_DEMENTIA"

print("Harmonised longitudinal diagnosis distribution")
print("----------------------------------------------")
print(
    longitudinal_dx["HARMONISED_DIAGNOSIS"]
    .value_counts(dropna=False)
)

print("\nDementia-event classification by phase")
print("--------------------------------------")
display(
    pd.crosstab(
        longitudinal_dx.loc[
            longitudinal_dx["DIAGNOSIS"].eq(3),
            "PHASE"
        ],
        longitudinal_dx.loc[
            longitudinal_dx["DIAGNOSIS"].eq(3),
            "HARMONISED_DIAGNOSIS"
        ],
        margins=True,
    )
)

print("\nBaseline diagnosis distribution after harmonisation")
print("---------------------------------------------------")
print(
    longitudinal_dx.loc[
        longitudinal_dx["VISCODE2"].eq("bl"),
        "HARMONISED_DIAGNOSIS"
    ].value_counts(dropna=False)
)

### 12.1. Harmonised diagnosis summary

The phase-specific rules produce a clean baseline distribution of:

- 1,196 CN;
- 1,286 MCI;
- 459 confirmed AD;
- 3 other-dementia participants.

The three baseline `OTHER_DEMENTIA` participants will be excluded from the four-group cohort. The 1,286 baseline-MCI participants will now be evaluated longitudinally to assign `sMCI`, `pMCI`, or an exclusion label.

# 13. Prepare longitudinal diagnosis histories for baseline-MCI participants

The next step is to isolate participants diagnosed with MCI at the formal baseline visit and organise all valid diagnosis events occurring on or after their baseline date.

Before applying the 36-month trajectory rules, this step will check for:

- diagnosis events occurring before the participant's formal baseline;
- multiple diagnosis records on the same date;
- conflicting diagnoses recorded on the same date;
- the number and duration of follow-up events available for each baseline-MCI participant.

In [ ]:
# Prepare longitudinal histories for all formal baseline-MCI participants

baseline_mci = (
    longitudinal_dx.loc[
        longitudinal_dx["VISCODE2"].eq("bl")
        & longitudinal_dx["HARMONISED_DIAGNOSIS"].eq("MCI"),
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
        ],
    ]
    .rename(
        columns={
            "PHASE": "BASELINE_PHASE",
            "EXAMDATE": "BASELINE_DATE",
        }
    )
    .sort_values("RID")
    .reset_index(drop=True)
)

mci_longitudinal = longitudinal_dx.merge(
    baseline_mci[
        [
            "RID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ],
    on="RID",
    how="inner",
)

mci_longitudinal["DAYS_FROM_BASELINE"] = (
    mci_longitudinal["EXAMDATE"]
    - mci_longitudinal["BASELINE_DATE"]
).dt.days

mci_followup = (
    mci_longitudinal.loc[
        mci_longitudinal["DAYS_FROM_BASELINE"] >= 0
    ]
    .copy()
    .sort_values(
        [
            "RID",
            "EXAMDATE",
            "VISCODE2",
            "HARMONISED_DIAGNOSIS",
        ]
    )
)

same_day_diagnosis_counts = (
    mci_followup
    .groupby(
        [
            "RID",
            "EXAMDATE",
        ]
    )["HARMONISED_DIAGNOSIS"]
    .nunique()
    .reset_index(name="SAME_DAY_DIAGNOSIS_COUNT")
)

same_day_conflicts = same_day_diagnosis_counts.loc[
    same_day_diagnosis_counts["SAME_DAY_DIAGNOSIS_COUNT"] > 1
].copy()

mci_followup_summary = (
    mci_followup
    .groupby("RID", as_index=False)
    .agg(
        FOLLOWUP_EVENT_COUNT=("EXAMDATE", "size"),
        UNIQUE_FOLLOWUP_DATES=("EXAMDATE", "nunique"),
        LAST_DIAGNOSIS_DATE=("EXAMDATE", "max"),
        MAX_DAYS_FROM_BASELINE=("DAYS_FROM_BASELINE", "max"),
    )
)

print("Baseline-MCI longitudinal preparation")
print("------------------------------------")
print(f"Baseline-MCI participants: {len(baseline_mci):,}")
print(f"Participants represented after baseline filtering: {mci_followup['RID'].nunique():,}")
print(
    "Diagnosis events before formal baseline:",
    f"{(mci_longitudinal['DAYS_FROM_BASELINE'] < 0).sum():,}"
)
print(
    "Participant-dates with conflicting same-day diagnoses:",
    f"{len(same_day_conflicts):,}"
)
print(
    "Participants with at least one same-day diagnosis conflict:",
    f"{same_day_conflicts['RID'].nunique():,}"
)

print("\nFollow-up duration in days")
print("--------------------------")
print(
    mci_followup_summary["MAX_DAYS_FROM_BASELINE"]
    .describe()
)

if len(same_day_conflicts) > 0:
    print("\nConflicting same-day diagnosis records")
    print("--------------------------------------")

    display(
        mci_followup.merge(
            same_day_conflicts[
                [
                    "RID",
                    "EXAMDATE",
                ]
            ],
            on=[
                "RID",
                "EXAMDATE",
            ],
            how="inner",
        )[
            [
                "RID",
                "PTID",
                "EXAMDATE",
                "VISCODE2",
                "HARMONISED_DIAGNOSIS",
                "DAYS_FROM_BASELINE",
            ]
        ].sort_values(
            [
                "RID",
                "EXAMDATE",
                "HARMONISED_DIAGNOSIS",
            ]
        )
    )

### 13.1. Baseline-MCI follow-up audit summary

All 1,286 baseline-MCI participants are represented in the longitudinal diagnosis history, with no conflicting diagnoses recorded on the same date.

The 756 diagnosis events occurring before formal baseline will not be used for prognosis classification. Follow-up duration varies substantially, so the 36-month trajectory rules must now distinguish valid sMCI and pMCI cases from short follow-up, late conversion, reversion, and unstable trajectories.

# 14. Assign 36-month MCI trajectory labels

Each baseline-MCI participant will now be classified using diagnosis events on or after the formal baseline date.

The rules are:

- `pMCI`: first confirmed AD diagnosis occurs within 36 months, with no CN diagnosis before conversion and no later reversion from AD;
- `sMCI`: remains MCI and has an MCI diagnosis at or after 36 months, with no AD or CN diagnosis;
- `exclude_short_followup`: no diagnosis evidence extending to 36 months;
- `exclude_late_converter`: first confirmed AD diagnosis occurs after 36 months;
- `exclude_reverter_to_cn`: a CN diagnosis occurs after baseline;
- `exclude_reverter_after_ad`: an AD diagnosis is followed by a later non-AD diagnosis;
- `exclude_unstable_before_conversion`: diagnosis changes before a valid AD conversion make the trajectory unreliable.

In [ ]:
# Assign participant-level 36-month MCI trajectory labels

OUTCOME_DAYS = 365.25 * 3

trajectory_rows = []

for rid, group in mci_followup.groupby("RID"):
    group = (
        group.sort_values(["EXAMDATE", "HARMONISED_DIAGNOSIS"])
        .drop_duplicates(
            subset=["EXAMDATE", "HARMONISED_DIAGNOSIS"],
            keep="first",
        )
        .copy()
    )

    baseline_row = baseline_mci.loc[
        baseline_mci["RID"].eq(rid)
    ].iloc[0]

    baseline_date = baseline_row["BASELINE_DATE"]
    outcome_window_end = baseline_date + pd.Timedelta(days=OUTCOME_DAYS)

    post_baseline = group.loc[
        group["EXAMDATE"] > baseline_date
    ].copy()

    ad_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("AD")
    ]

    cn_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("CN")
    ]

    mci_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("MCI")
    ]

    other_dementia_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("OTHER_DEMENTIA")
    ]

    first_ad_date = (
        ad_events["EXAMDATE"].min()
        if not ad_events.empty
        else pd.NaT
    )

    first_cn_date = (
        cn_events["EXAMDATE"].min()
        if not cn_events.empty
        else pd.NaT
    )

    first_mci_at_or_after_36m = (
        mci_events.loc[
            mci_events["EXAMDATE"] >= outcome_window_end,
            "EXAMDATE",
        ].min()
        if not mci_events.empty
        else pd.NaT
    )

    last_diagnosis_date = group["EXAMDATE"].max()
    last_diagnosis = group.iloc[-1]["HARMONISED_DIAGNOSIS"]

    ad_within_36m = (
        pd.notna(first_ad_date)
        and first_ad_date <= outcome_window_end
    )

    ad_after_36m = (
        pd.notna(first_ad_date)
        and first_ad_date > outcome_window_end
    )

    has_mci_at_or_after_36m = pd.notna(
        first_mci_at_or_after_36m
    )

    has_any_diagnosis_at_or_after_36m = (
        last_diagnosis_date >= outcome_window_end
    )

    has_cn_after_baseline = not cn_events.empty
    has_other_dementia_after_baseline = (
        not other_dementia_events.empty
    )

    has_non_ad_after_first_ad = False

    if pd.notna(first_ad_date):
        has_non_ad_after_first_ad = (
            post_baseline.loc[
                post_baseline["EXAMDATE"] > first_ad_date,
                "HARMONISED_DIAGNOSIS",
            ]
            .ne("AD")
            .any()
        )

    has_unstable_before_conversion = False

    if pd.notna(first_ad_date):
        pre_conversion_diagnoses = post_baseline.loc[
            post_baseline["EXAMDATE"] < first_ad_date,
            "HARMONISED_DIAGNOSIS",
        ]

        has_unstable_before_conversion = (
            pre_conversion_diagnoses
            .isin(["CN", "OTHER_DEMENTIA"])
            .any()
        )

    if has_cn_after_baseline:
        trajectory_label = "exclude_reverter_to_cn"

    elif has_other_dementia_after_baseline:
        trajectory_label = "exclude_unstable_before_conversion"

    elif ad_within_36m and has_non_ad_after_first_ad:
        trajectory_label = "exclude_reverter_after_ad"

    elif ad_within_36m and has_unstable_before_conversion:
        trajectory_label = "exclude_unstable_before_conversion"

    elif ad_within_36m:
        trajectory_label = "pMCI"

    elif ad_after_36m:
        trajectory_label = "exclude_late_converter"

    elif has_mci_at_or_after_36m:
        trajectory_label = "sMCI"

    else:
        trajectory_label = "exclude_short_followup"

    prognosis_target = {
        "sMCI": 0,
        "pMCI": 1,
    }.get(trajectory_label, pd.NA)

    trajectory_rows.append(
        {
            "RID": rid,
            "PTID": baseline_row["PTID"],
            "BASELINE_PHASE": baseline_row["BASELINE_PHASE"],
            "BASELINE_DATE": baseline_date,
            "OUTCOME_WINDOW_END": outcome_window_end,
            "FOLLOWUP_EVENT_COUNT": len(post_baseline),
            "LAST_DIAGNOSIS_DATE": last_diagnosis_date,
            "LAST_DIAGNOSIS": last_diagnosis,
            "FIRST_AD_DATE": first_ad_date,
            "FIRST_CN_DATE": first_cn_date,
            "FIRST_MCI_AT_OR_AFTER_36M": first_mci_at_or_after_36m,
            "AD_WITHIN_36M": ad_within_36m,
            "HAS_MCI_AT_OR_AFTER_36M": has_mci_at_or_after_36m,
            "HAS_ANY_DIAGNOSIS_AT_OR_AFTER_36M": (
                has_any_diagnosis_at_or_after_36m
            ),
            "HAS_NON_AD_AFTER_FIRST_AD": has_non_ad_after_first_ad,
            "HAS_OTHER_DEMENTIA_AFTER_BASELINE": (
                has_other_dementia_after_baseline
            ),
            "TRAJECTORY_LABEL": trajectory_label,
            "PROGNOSIS_TARGET": prognosis_target,
        }
    )

mci_trajectory_reconstructed = pd.DataFrame(trajectory_rows)

print("Reconstructed MCI trajectory labels")
print("-----------------------------------")
print(f"Rows: {len(mci_trajectory_reconstructed):,}")
print(f"Unique RID: {mci_trajectory_reconstructed['RID'].nunique():,}")

print("\nTrajectory label distribution:")
print(
    mci_trajectory_reconstructed["TRAJECTORY_LABEL"]
    .value_counts(dropna=False)
)

### 14.1. Reconstructed trajectory result

The reconstructed labels differ from the earlier trajectory manifest:

- `pMCI` decreased from 259 to 249;
- `exclude_unstable_before_conversion` increased because dementia events not explicitly confirmed as Alzheimer’s disease are now treated as `OTHER_DEMENTIA`;
- several other exclusion counts also changed.

Before accepting this reconstruction, the new participant-level labels must be compared directly with the previous trajectory manifest to identify exactly which participants changed and why.

In [ ]:
# Compare reconstructed trajectory labels with the previous saved manifest

previous_trajectory = mci_trajectory_labels[
    ["RID", "TRAJECTORY_LABEL", "PROGNOSIS_TARGET"]
].copy()

previous_trajectory = previous_trajectory.rename(
    columns={
        "TRAJECTORY_LABEL": "PREVIOUS_TRAJECTORY_LABEL",
        "PROGNOSIS_TARGET": "PREVIOUS_PROGNOSIS_TARGET",
    }
)

trajectory_comparison = mci_trajectory_reconstructed.merge(
    previous_trajectory,
    on="RID",
    how="outer",
    validate="one_to_one",
)

trajectory_comparison["LABEL_CHANGED"] = (
    trajectory_comparison["TRAJECTORY_LABEL"]
    != trajectory_comparison["PREVIOUS_TRAJECTORY_LABEL"]
)

print("Trajectory reconstruction comparison")
print("------------------------------------")
print(f"Participants compared: {len(trajectory_comparison):,}")
print(
    "Unchanged labels:",
    f"{(~trajectory_comparison['LABEL_CHANGED']).sum():,}"
)
print(
    "Changed labels:",
    f"{trajectory_comparison['LABEL_CHANGED'].sum():,}"
)

print("\nPrevious-to-reconstructed label transitions:")
display(
    pd.crosstab(
        trajectory_comparison["PREVIOUS_TRAJECTORY_LABEL"],
        trajectory_comparison["TRAJECTORY_LABEL"],
        margins=True,
    )
)

print("\nParticipants whose trajectory label changed:")
display(
    trajectory_comparison.loc[
        trajectory_comparison["LABEL_CHANGED"],
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "FIRST_AD_DATE",
            "FIRST_CN_DATE",
            "LAST_DIAGNOSIS_DATE",
            "LAST_DIAGNOSIS",
            "HAS_NON_AD_AFTER_FIRST_AD",
            "HAS_OTHER_DEMENTIA_AFTER_BASELINE",
            "PREVIOUS_TRAJECTORY_LABEL",
            "TRAJECTORY_LABEL",
        ],
    ].sort_values(
        [
            "PREVIOUS_TRAJECTORY_LABEL",
            "TRAJECTORY_LABEL",
            "RID",
        ]
    )
)

### 14.2. Interpretation of changed trajectory labels

The comparison shows that the new phase-specific AD confirmation is working, but the current trajectory code is too broad when handling `OTHER_DEMENTIA`.

At present, any `OTHER_DEMENTIA` event after baseline causes:

`exclude_unstable_before_conversion`

even when that event occurred:

- after a confirmed AD conversion;
- years after the 36-month outcome window;
- without any confirmed AD conversion at all.

Therefore, the 23 participants assigned to `exclude_unstable_before_conversion` must be reviewed chronologically before the rules are finalised. The next audit will distinguish:

- `OTHER_DEMENTIA` before confirmed AD conversion;
- `OTHER_DEMENTIA` after confirmed AD conversion;
- `OTHER_DEMENTIA` without any confirmed AD event;
- events occurring within versus after the 36-month outcome window.

In [ ]:
# Audit the chronological position of OTHER_DEMENTIA events

unstable_rids = set(
    mci_trajectory_reconstructed.loc[
        mci_trajectory_reconstructed["TRAJECTORY_LABEL"].eq(
            "exclude_unstable_before_conversion"
        ),
        "RID",
    ].astype(int)
)

unstable_event_rows = []

for rid in sorted(unstable_rids):
    group = (
        mci_followup.loc[
            mci_followup["RID"].eq(rid)
        ]
        .sort_values(["EXAMDATE", "HARMONISED_DIAGNOSIS"])
        .copy()
    )

    baseline_date = group["BASELINE_DATE"].iloc[0]
    outcome_window_end = baseline_date + pd.Timedelta(days=365.25 * 3)

    first_ad_date = group.loc[
        (group["EXAMDATE"] > baseline_date)
        & group["HARMONISED_DIAGNOSIS"].eq("AD"),
        "EXAMDATE",
    ].min()

    other_events = group.loc[
        (group["EXAMDATE"] > baseline_date)
        & group["HARMONISED_DIAGNOSIS"].eq("OTHER_DEMENTIA")
    ]

    for _, row in other_events.iterrows():
        if pd.isna(first_ad_date):
            relation_to_ad = "no_confirmed_ad"
        elif row["EXAMDATE"] < first_ad_date:
            relation_to_ad = "before_first_ad"
        elif row["EXAMDATE"] == first_ad_date:
            relation_to_ad = "same_day_as_first_ad"
        else:
            relation_to_ad = "after_first_ad"

        unstable_event_rows.append(
            {
                "RID": rid,
                "PTID": row["PTID"],
                "BASELINE_PHASE": row["BASELINE_PHASE"],
                "BASELINE_DATE": baseline_date,
                "OUTCOME_WINDOW_END": outcome_window_end,
                "FIRST_AD_DATE": first_ad_date,
                "OTHER_DEMENTIA_DATE": row["EXAMDATE"],
                "OTHER_DEMENTIA_VISCODE2": row["VISCODE2"],
                "DAYS_FROM_BASELINE": row["DAYS_FROM_BASELINE"],
                "WITHIN_36M": row["EXAMDATE"] <= outcome_window_end,
                "RELATION_TO_FIRST_AD": relation_to_ad,
            }
        )

unstable_chronology_audit = pd.DataFrame(unstable_event_rows)

print("OTHER_DEMENTIA chronology audit")
print("--------------------------------")
print(f"Participants reviewed: {len(unstable_rids):,}")
print(f"OTHER_DEMENTIA events: {len(unstable_chronology_audit):,}")

print("\nRelationship to first confirmed AD:")
print(
    unstable_chronology_audit["RELATION_TO_FIRST_AD"]
    .value_counts(dropna=False)
)

print("\nTiming relative to the 36-month outcome window:")
print(
    unstable_chronology_audit["WITHIN_36M"]
    .value_counts(dropna=False)
)

print("\nParticipant-level chronology:")
display(
    unstable_chronology_audit.sort_values(
        ["RELATION_TO_FIRST_AD", "RID", "OTHER_DEMENTIA_DATE"]
    )
)

### 14.3. Chronology audit decision

The audit shows that `OTHER_DEMENTIA` should not automatically override every later trajectory.

The corrected priority will be:

- if confirmed AD occurs within 36 months and is followed by any later non-AD diagnosis, classify as `exclude_reverter_after_ad`;
- if `OTHER_DEMENTIA` occurs before the first confirmed AD, classify as `exclude_unstable_before_conversion`;
- if no confirmed AD occurs but `OTHER_DEMENTIA` appears after baseline, classify as `exclude_unstable_before_conversion`;
- otherwise, apply the usual pMCI, late-converter, sMCI, and short-follow-up rules.

The 36-month boundary will also be calculated using exactly three calendar years rather than an approximate number of days.

In [ ]:
# Reassign MCI trajectories using chronology-aware diagnosis rules

trajectory_rows = []

for rid, group in mci_followup.groupby("RID"):
    group = (
        group.sort_values(["EXAMDATE", "HARMONISED_DIAGNOSIS"])
        .drop_duplicates(
            subset=["EXAMDATE", "HARMONISED_DIAGNOSIS"],
            keep="first",
        )
        .copy()
    )

    baseline_row = baseline_mci.loc[
        baseline_mci["RID"].eq(rid)
    ].iloc[0]

    baseline_date = baseline_row["BASELINE_DATE"]
    outcome_window_end = baseline_date + pd.DateOffset(years=3)

    post_baseline = group.loc[
        group["EXAMDATE"] > baseline_date
    ].copy()

    ad_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("AD")
    ]

    cn_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("CN")
    ]

    mci_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("MCI")
    ]

    other_dementia_events = post_baseline.loc[
        post_baseline["HARMONISED_DIAGNOSIS"].eq("OTHER_DEMENTIA")
    ]

    first_ad_date = (
        ad_events["EXAMDATE"].min()
        if not ad_events.empty
        else pd.NaT
    )

    first_cn_date = (
        cn_events["EXAMDATE"].min()
        if not cn_events.empty
        else pd.NaT
    )

    first_other_dementia_date = (
        other_dementia_events["EXAMDATE"].min()
        if not other_dementia_events.empty
        else pd.NaT
    )

    first_mci_at_or_after_36m = (
        mci_events.loc[
            mci_events["EXAMDATE"] >= outcome_window_end,
            "EXAMDATE",
        ].min()
        if not mci_events.empty
        else pd.NaT
    )

    last_diagnosis_date = group["EXAMDATE"].max()
    last_diagnosis = group.iloc[-1]["HARMONISED_DIAGNOSIS"]

    ad_within_36m = (
        pd.notna(first_ad_date)
        and first_ad_date <= outcome_window_end
    )

    ad_after_36m = (
        pd.notna(first_ad_date)
        and first_ad_date > outcome_window_end
    )

    has_mci_at_or_after_36m = pd.notna(
        first_mci_at_or_after_36m
    )

    has_any_diagnosis_at_or_after_36m = (
        last_diagnosis_date >= outcome_window_end
    )

    has_cn_after_baseline = not cn_events.empty

    has_other_dementia_after_baseline = (
        not other_dementia_events.empty
    )

    has_other_dementia_before_first_ad = (
        pd.notna(first_ad_date)
        and pd.notna(first_other_dementia_date)
        and first_other_dementia_date < first_ad_date
    )

    has_non_ad_after_first_ad = False

    if pd.notna(first_ad_date):
        has_non_ad_after_first_ad = (
            post_baseline.loc[
                post_baseline["EXAMDATE"] > first_ad_date,
                "HARMONISED_DIAGNOSIS",
            ]
            .ne("AD")
            .any()
        )

    if has_cn_after_baseline:
        trajectory_label = "exclude_reverter_to_cn"

    elif pd.notna(first_ad_date) and has_non_ad_after_first_ad:
        trajectory_label = "exclude_reverter_after_ad"

    elif has_other_dementia_before_first_ad:
        trajectory_label = "exclude_unstable_before_conversion"

    elif pd.isna(first_ad_date) and has_other_dementia_after_baseline:
        trajectory_label = "exclude_unstable_before_conversion"

    elif ad_within_36m:
        trajectory_label = "pMCI"

    elif ad_after_36m:
        trajectory_label = "exclude_late_converter"

    elif has_mci_at_or_after_36m:
        trajectory_label = "sMCI"

    else:
        trajectory_label = "exclude_short_followup"

    prognosis_target = {
        "sMCI": 0,
        "pMCI": 1,
    }.get(trajectory_label, pd.NA)

    trajectory_rows.append(
        {
            "RID": rid,
            "PTID": baseline_row["PTID"],
            "BASELINE_PHASE": baseline_row["BASELINE_PHASE"],
            "BASELINE_DATE": baseline_date,
            "OUTCOME_WINDOW_END": outcome_window_end,
            "FOLLOWUP_EVENT_COUNT": len(post_baseline),
            "LAST_DIAGNOSIS_DATE": last_diagnosis_date,
            "LAST_DIAGNOSIS": last_diagnosis,
            "FIRST_AD_DATE": first_ad_date,
            "FIRST_CN_DATE": first_cn_date,
            "FIRST_OTHER_DEMENTIA_DATE": first_other_dementia_date,
            "FIRST_MCI_AT_OR_AFTER_36M": first_mci_at_or_after_36m,
            "AD_WITHIN_36M": ad_within_36m,
            "HAS_MCI_AT_OR_AFTER_36M": has_mci_at_or_after_36m,
            "HAS_ANY_DIAGNOSIS_AT_OR_AFTER_36M": (
                has_any_diagnosis_at_or_after_36m
            ),
            "HAS_NON_AD_AFTER_FIRST_AD": has_non_ad_after_first_ad,
            "HAS_OTHER_DEMENTIA_AFTER_BASELINE": (
                has_other_dementia_after_baseline
            ),
            "HAS_OTHER_DEMENTIA_BEFORE_FIRST_AD": (
                has_other_dementia_before_first_ad
            ),
            "TRAJECTORY_LABEL": trajectory_label,
            "PROGNOSIS_TARGET": prognosis_target,
        }
    )

mci_trajectory_reconstructed = pd.DataFrame(trajectory_rows)

print("Chronology-aware MCI trajectory labels")
print("--------------------------------------")
print(f"Rows: {len(mci_trajectory_reconstructed):,}")
print(f"Unique RID: {mci_trajectory_reconstructed['RID'].nunique():,}")

print("\nTrajectory label distribution:")
print(
    mci_trajectory_reconstructed["TRAJECTORY_LABEL"]
    .value_counts(dropna=False)
)

### 14.4. Final MCI trajectory distribution

The chronology-aware reconstruction produced:

- 295 sMCI;
- 249 pMCI;
- 117 late converters;
- 117 reverters to CN;
- 28 reverters after AD;
- 17 unstable trajectories involving other dementia;
- 463 participants with insufficient follow-up.

Before saving this as the authoritative trajectory manifest, the 17 unstable cases and 28 post-AD reverters should remain excluded. The final prognosis cohort therefore contains 544 baseline-MCI participants.

# 15. Construct the authoritative four-group clinical cohort

The final clinical cohort will now combine:

- baseline CN participants;
- baseline confirmed-AD participants;
- reconstructed sMCI participants;
- reconstructed pMCI participants.

Baseline `OTHER_DEMENTIA` participants and all excluded MCI trajectories will remain outside the modelling cohort. The resulting manifest will contain one row per participant and will preserve the baseline date, cohort label, prognosis target, and key longitudinal outcome dates.

In [ ]:
# Construct the authoritative four-group clinical cohort

baseline_cn = (
    longitudinal_dx.loc[
        longitudinal_dx["VISCODE2"].eq("bl")
        & longitudinal_dx["HARMONISED_DIAGNOSIS"].eq("CN"),
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
        ],
    ]
    .rename(
        columns={
            "PHASE": "BASELINE_PHASE",
            "EXAMDATE": "BASELINE_DATE",
        }
    )
    .assign(
        COHORT_LABEL="CN",
        PROGNOSIS_TARGET=pd.NA,
        FIRST_AD_DATE=pd.NaT,
        LAST_DIAGNOSIS_DATE=pd.NaT,
        TRAJECTORY_LABEL=pd.NA,
    )
)

baseline_ad = (
    longitudinal_dx.loc[
        longitudinal_dx["VISCODE2"].eq("bl")
        & longitudinal_dx["HARMONISED_DIAGNOSIS"].eq("AD"),
        [
            "RID",
            "PTID",
            "PHASE",
            "EXAMDATE",
        ],
    ]
    .rename(
        columns={
            "PHASE": "BASELINE_PHASE",
            "EXAMDATE": "BASELINE_DATE",
        }
    )
    .assign(
        COHORT_LABEL="AD",
        PROGNOSIS_TARGET=pd.NA,
        FIRST_AD_DATE=pd.NaT,
        LAST_DIAGNOSIS_DATE=pd.NaT,
        TRAJECTORY_LABEL=pd.NA,
    )
)

final_mci = (
    mci_trajectory_reconstructed.loc[
        mci_trajectory_reconstructed["TRAJECTORY_LABEL"].isin(
            ["sMCI", "pMCI"]
        ),
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "FIRST_AD_DATE",
            "LAST_DIAGNOSIS_DATE",
            "TRAJECTORY_LABEL",
            "PROGNOSIS_TARGET",
        ],
    ]
    .copy()
    .rename(
        columns={
            "TRAJECTORY_LABEL": "COHORT_LABEL",
        }
    )
)

final_mci["TRAJECTORY_LABEL"] = final_mci["COHORT_LABEL"]

authoritative_clinical_cohort = pd.concat(
    [
        baseline_cn,
        baseline_ad,
        final_mci,
    ],
    ignore_index=True,
    sort=False,
)

authoritative_clinical_cohort = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "PROGNOSIS_TARGET",
            "TRAJECTORY_LABEL",
            "FIRST_AD_DATE",
            "LAST_DIAGNOSIS_DATE",
        ]
    ]
    .sort_values(["COHORT_LABEL", "RID"])
    .reset_index(drop=True)
)

print("Authoritative four-group clinical cohort")
print("----------------------------------------")
print(f"Rows: {len(authoritative_clinical_cohort):,}")
print(f"Unique RID: {authoritative_clinical_cohort['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{authoritative_clinical_cohort['RID'].duplicated().sum():,}"
)

print("\nCohort distribution:")
print(
    authoritative_clinical_cohort["COHORT_LABEL"]
    .value_counts(dropna=False)
)

display(authoritative_clinical_cohort.head())

### 15.1. Authoritative cohort summary

The reconstructed four-group cohort contains 2,199 unique participants:

- 1,196 CN;
- 459 confirmed AD;
- 295 sMCI;
- 249 pMCI.

All participants have one formal DXSUM baseline record. The AD group includes only phase-specifically confirmed Alzheimer’s disease cases, while the MCI groups use the chronology-aware 36-month trajectory rules.

# 16. Save the authoritative clinical cohort manifests

The reconstructed cohort and its supporting trajectory tables will now be saved in the dedicated `authoritative_clinical_cohort` folder.

Separate files will be retained for:

- the final four-group modelling cohort;
- all reconstructed baseline-MCI trajectory outcomes;
- excluded baseline-MCI participants;
- baseline participants excluded because their dementia was not confirmed as Alzheimer’s disease.

In [ ]:
# Save the authoritative clinical cohort and supporting manifests

FINAL_COHORT_PATH = (
    CLINICAL_COHORT_DIR
    / "authoritative_four_group_clinical_cohort.csv"
)

MCI_TRAJECTORY_PATH = (
    CLINICAL_COHORT_DIR
    / "authoritative_mci_36m_trajectory_labels.csv"
)

MCI_EXCLUSIONS_PATH = (
    CLINICAL_COHORT_DIR
    / "authoritative_mci_trajectory_exclusions.csv"
)

BASELINE_OTHER_DEMENTIA_PATH = (
    CLINICAL_COHORT_DIR
    / "excluded_baseline_other_dementia.csv"
)

authoritative_clinical_cohort.to_csv(
    FINAL_COHORT_PATH,
    index=False,
)

mci_trajectory_reconstructed.to_csv(
    MCI_TRAJECTORY_PATH,
    index=False,
)

mci_trajectory_reconstructed.loc[
    ~mci_trajectory_reconstructed["TRAJECTORY_LABEL"].isin(
        ["sMCI", "pMCI"]
    )
].to_csv(
    MCI_EXCLUSIONS_PATH,
    index=False,
)

baseline_dementia.loc[
    ~baseline_dementia["CONFIRMED_AD_OFFICIAL"]
].to_csv(
    BASELINE_OTHER_DEMENTIA_PATH,
    index=False,
)

saved_files = [
    FINAL_COHORT_PATH,
    MCI_TRAJECTORY_PATH,
    MCI_EXCLUSIONS_PATH,
    BASELINE_OTHER_DEMENTIA_PATH,
]

print("Saved authoritative cohort files")
print("--------------------------------")

for path in saved_files:
    print(f"{path.name}: {'SAVED' if path.exists() else 'FAILED'}")

print("\nOutput folder:")
print(CLINICAL_COHORT_DIR)

# 17. Align PTDEMOG with the authoritative clinical cohort

The cleaned longitudinal PTDEMOG table will be matched to the reconstructed four-group clinical cohort.

Because demographic variables are generally stable or collected once, PTDEMOG does not require the same strict temporal window as cognitive tests or biomarkers. However, the participant record selected for modelling must still be traceable to the clinical baseline.

This step will first load:

- the authoritative four-group clinical cohort;
- the cleaned longitudinal PTDEMOG table.

The available identifiers, visit fields, dates, and demographic variables will then be inspected before defining the participant-level selection rule.

In [ ]:
# Load the authoritative cohort and cleaned PTDEMOG table

FINAL_COHORT_PATH = (
    CLINICAL_COHORT_DIR
    / "authoritative_four_group_clinical_cohort.csv"
)

PTDEMOG_PATH = (
    INTERIM_DIR
    / "ptdemog_cleaned_longitudinal.csv"
)

authoritative_clinical_cohort = pd.read_csv(
    FINAL_COHORT_PATH,
    low_memory=False,
)

ptdemog_cleaned = pd.read_csv(
    PTDEMOG_PATH,
    low_memory=False,
)

print("Authoritative clinical cohort")
print("-----------------------------")
print("Path:", FINAL_COHORT_PATH)
print("Exists:", FINAL_COHORT_PATH.exists())
print(f"Rows: {len(authoritative_clinical_cohort):,}")
print(f"Columns: {len(authoritative_clinical_cohort.columns)}")
print(authoritative_clinical_cohort.columns.tolist())

print("\nCleaned PTDEMOG")
print("---------------")
print("Path:", PTDEMOG_PATH)
print("Exists:", PTDEMOG_PATH.exists())
print(f"Rows: {len(ptdemog_cleaned):,}")
print(f"Columns: {len(ptdemog_cleaned.columns)}")
print(ptdemog_cleaned.columns.tolist())

# 18. Audit PTDEMOG coverage and timing relative to clinical baseline

PTDEMOG contains multiple longitudinal records per participant. Before selecting one modelling record, the table must be matched to the authoritative cohort and its available dates assessed.

This step will determine:

- how many cohort participants appear in PTDEMOG;
- how many have at least one valid PTDEMOG visit date;
- how many have a formal PTDEMOG baseline record;
- the distance between each PTDEMOG record and the DXSUM clinical baseline date;
- whether any participants have only undated demographic records.

No participant-level record will be selected yet.

In [ ]:
# Standardise identifiers and dates, then audit PTDEMOG coverage

authoritative_clinical_cohort["RID"] = pd.to_numeric(
    authoritative_clinical_cohort["RID"],
    errors="coerce"
).astype("Int64")

authoritative_clinical_cohort["BASELINE_DATE"] = pd.to_datetime(
    authoritative_clinical_cohort["BASELINE_DATE"],
    errors="coerce"
)

ptdemog_cleaned["RID_CLEAN"] = pd.to_numeric(
    ptdemog_cleaned["RID_CLEAN"],
    errors="coerce"
).astype("Int64")

ptdemog_cleaned["VISDATE_PARSED"] = pd.to_datetime(
    ptdemog_cleaned["VISDATE_PARSED"],
    errors="coerce"
)

ptdemog_cohort = ptdemog_cleaned.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
        ]
    ],
    left_on="RID_CLEAN",
    right_on="RID",
    how="inner",
    suffixes=("_PTDEMOG", "_COHORT"),
)

ptdemog_cohort["DAYS_FROM_BASELINE"] = (
    ptdemog_cohort["VISDATE_PARSED"]
    - ptdemog_cohort["BASELINE_DATE"]
).dt.days

ptdemog_cohort["ABS_DAYS_FROM_BASELINE"] = (
    ptdemog_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

ptdemog_rids = set(
    ptdemog_cleaned["RID_CLEAN"]
    .dropna()
    .astype(int)
)

matched_rids = cohort_rids & ptdemog_rids

participant_ptdemog_audit = (
    ptdemog_cohort
    .groupby("RID", as_index=False)
    .agg(
        PTDEMOG_ROW_COUNT=("RID", "size"),
        VALID_DATED_ROW_COUNT=("VISDATE_PARSED", lambda x: x.notna().sum()),
        HAS_FORMAL_BL_RECORD=(
            "VISCODE2_CLEAN",
            lambda x: x.astype("string").eq("bl").any()
        ),
        MIN_ABS_DAYS_FROM_BASELINE=("ABS_DAYS_FROM_BASELINE", "min"),
    )
)

print("PTDEMOG cohort coverage")
print("-----------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in PTDEMOG: {len(matched_rids):,}")
print(f"Participants absent from PTDEMOG: {len(cohort_rids - ptdemog_rids):,}")

print("\nMatched participant date availability")
print("-------------------------------------")
print(
    "With at least one valid dated PTDEMOG record:",
    f"{(participant_ptdemog_audit['VALID_DATED_ROW_COUNT'] > 0).sum():,}"
)
print(
    "With no valid dated PTDEMOG record:",
    f"{(participant_ptdemog_audit['VALID_DATED_ROW_COUNT'] == 0).sum():,}"
)
print(
    "With a formal PTDEMOG baseline record:",
    f"{participant_ptdemog_audit['HAS_FORMAL_BL_RECORD'].sum():,}"
)

print("\nNearest dated PTDEMOG record to clinical baseline")
print("-------------------------------------------------")
print(
    participant_ptdemog_audit["MIN_ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nCoverage by cohort label")
print("------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_PTDEMOG=authoritative_clinical_cohort["RID"].isin(matched_rids)
    )
    .groupby("COHORT_LABEL")["HAS_PTDEMOG"]
    .value_counts()
    .unstack(fill_value=0)
)

### 18.1. PTDEMOG audit error

The merge created suffixed RID columns because both tables already contained a column named `RID`. As a result, there was no column named exactly `RID` available for grouping.

The corrected cell below renames the cohort identifier before merging, so the matched participant ID is unambiguous.

In [ ]:
# Standardise identifiers and dates, then audit PTDEMOG coverage

authoritative_clinical_cohort["RID"] = pd.to_numeric(
    authoritative_clinical_cohort["RID"],
    errors="coerce"
).astype("Int64")

authoritative_clinical_cohort["BASELINE_DATE"] = pd.to_datetime(
    authoritative_clinical_cohort["BASELINE_DATE"],
    errors="coerce"
)

ptdemog_cleaned["RID_CLEAN"] = pd.to_numeric(
    ptdemog_cleaned["RID_CLEAN"],
    errors="coerce"
).astype("Int64")

ptdemog_cleaned["VISDATE_PARSED"] = pd.to_datetime(
    ptdemog_cleaned["VISDATE_PARSED"],
    errors="coerce"
)

cohort_for_merge = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
        ]
    ]
    .rename(
        columns={
            "RID": "COHORT_RID",
            "PTID": "COHORT_PTID",
        }
    )
)

ptdemog_cohort = ptdemog_cleaned.merge(
    cohort_for_merge,
    left_on="RID_CLEAN",
    right_on="COHORT_RID",
    how="inner",
)

ptdemog_cohort["DAYS_FROM_BASELINE"] = (
    ptdemog_cohort["VISDATE_PARSED"]
    - ptdemog_cohort["BASELINE_DATE"]
).dt.days

ptdemog_cohort["ABS_DAYS_FROM_BASELINE"] = (
    ptdemog_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

ptdemog_rids = set(
    ptdemog_cleaned["RID_CLEAN"]
    .dropna()
    .astype(int)
)

matched_rids = cohort_rids & ptdemog_rids

participant_ptdemog_audit = (
    ptdemog_cohort
    .groupby("COHORT_RID", as_index=False)
    .agg(
        PTDEMOG_ROW_COUNT=("COHORT_RID", "size"),
        VALID_DATED_ROW_COUNT=(
            "VISDATE_PARSED",
            lambda x: x.notna().sum()
        ),
        HAS_FORMAL_BL_RECORD=(
            "VISCODE2_CLEAN",
            lambda x: x.astype("string").eq("bl").any()
        ),
        MIN_ABS_DAYS_FROM_BASELINE=(
            "ABS_DAYS_FROM_BASELINE",
            "min"
        ),
    )
)

print("PTDEMOG cohort coverage")
print("-----------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in PTDEMOG: {len(matched_rids):,}")
print(
    "Participants absent from PTDEMOG:",
    f"{len(cohort_rids - ptdemog_rids):,}"
)

print("\nMatched participant date availability")
print("-------------------------------------")
print(
    "With at least one valid dated PTDEMOG record:",
    f"{(participant_ptdemog_audit['VALID_DATED_ROW_COUNT'] > 0).sum():,}"
)
print(
    "With no valid dated PTDEMOG record:",
    f"{(participant_ptdemog_audit['VALID_DATED_ROW_COUNT'] == 0).sum():,}"
)
print(
    "With a formal PTDEMOG baseline record:",
    f"{participant_ptdemog_audit['HAS_FORMAL_BL_RECORD'].sum():,}"
)

print("\nNearest dated PTDEMOG record to clinical baseline")
print("-------------------------------------------------")
print(
    participant_ptdemog_audit["MIN_ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nCoverage by cohort label")
print("------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_PTDEMOG=authoritative_clinical_cohort["RID"].isin(
            matched_rids
        )
    )
    .groupby("COHORT_LABEL")["HAS_PTDEMOG"]
    .value_counts()
    .unstack(fill_value=0)
)

### 18.2. PTDEMOG coverage summary

PTDEMOG covers all 2,199 participants in the authoritative cohort, and every participant has at least one dated demographic record.

Only 7 participants have a PTDEMOG row labelled `bl`, so `VISCODE2 = bl` cannot be used as the main selection rule for this table. This is expected because demographic information is often collected at screening or enrolment rather than repeated at the formal diagnostic baseline.

Unlike cognitive scores or biomarkers, most PTDEMOG variables are stable characteristics and do not create outcome leakage merely because they were recorded before baseline. However, very distant records and records containing demographic conflicts should still be audited before selecting one participant-level row.

# 19. Audit the nearest PTDEMOG record for each participant

For each participant, the nearest dated PTDEMOG row to the DXSUM clinical baseline will be identified.

This audit will examine:

- whether the nearest row occurs before or after baseline;
- the visit-code distribution of selected rows;
- the number of participants within 90, 180, and 365 days;
- participants whose nearest demographic record is more than one year from baseline;
- whether the nearest row contains stable-demographic conflicts or QC errors.

No final PTDEMOG record will be saved until these cases are reviewed.

In [ ]:
# Identify and audit the nearest dated PTDEMOG row per participant

ptdemog_nearest = (
    ptdemog_cohort.loc[
        ptdemog_cohort["VISDATE_PARSED"].notna()
    ]
    .sort_values(
        [
            "COHORT_RID",
            "ABS_DAYS_FROM_BASELINE",
            "VISDATE_PARSED",
        ],
        ascending=[True, True, True],
    )
    .drop_duplicates(
        subset="COHORT_RID",
        keep="first",
    )
    .copy()
)

ptdemog_nearest["TIMING_DIRECTION"] = pd.cut(
    ptdemog_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

print("Nearest PTDEMOG record audit")
print("----------------------------")
print(f"Selected rows: {len(ptdemog_nearest):,}")
print(f"Unique participants: {ptdemog_nearest['COHORT_RID'].nunique():,}")
print(
    "Duplicate participants:",
    f"{ptdemog_nearest['COHORT_RID'].duplicated().sum():,}"
)

print("\nTiming direction")
print("----------------")
print(
    ptdemog_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nSelected PTDEMOG visit codes")
print("---------------------------")
print(
    ptdemog_nearest["VISCODE2_CLEAN"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [90, 180, 365]:
    count = ptdemog_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(ptdemog_nearest)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}%)"
    )

print(
    "More than 365 days away:",
    f"{ptdemog_nearest['ABS_DAYS_FROM_BASELINE'].gt(365).sum():,}"
)

print("\nConflict and QC flags in selected rows")
print("--------------------------------------")
print(
    "Any stable-demographic conflict:",
    f"{ptdemog_nearest['ANY_STABLE_DEMOGRAPHIC_CONFLICT'].fillna(False).sum():,}"
)
print(
    "High-priority demographic conflict:",
    f"{ptdemog_nearest['HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT'].fillna(False).sum():,}"
)
print(
    "QC error flagged:",
    f"{ptdemog_nearest['HAS_QC_ERROR_CLEAN'].fillna(False).sum():,}"
)

print("\nParticipants whose nearest PTDEMOG row is more than one year away")
print("----------------------------------------------------------------")
display(
    ptdemog_nearest.loc[
        ptdemog_nearest["ABS_DAYS_FROM_BASELINE"] > 365,
        [
            "COHORT_RID",
            "COHORT_PTID",
            "COHORT_LABEL",
            "BASELINE_DATE",
            "VISDATE_PARSED",
            "VISCODE2_CLEAN",
            "DAYS_FROM_BASELINE",
            "ABS_DAYS_FROM_BASELINE",
            "DEMOGRAPHIC_NON_MISSING_COUNT",
            "ANY_STABLE_DEMOGRAPHIC_CONFLICT",
            "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT",
            "HAS_QC_ERROR_CLEAN",
        ],
    ].sort_values(
        "ABS_DAYS_FROM_BASELINE",
        ascending=False,
    )
)

# 20. Inspect high-priority PTDEMOG conflicts

Five selected PTDEMOG records are flagged with high-priority demographic conflicts.

Before saving the participant-level PTDEMOG cohort, these participants will be inspected across all available demographic records to determine:

- which stable variables conflict;
- whether the nearest-to-baseline row contains the most reliable values;
- whether an alternative record closer to baseline resolves the conflict;
- whether any variable should be set to missing rather than forcing a value.

In [ ]:
# Inspect all PTDEMOG records for participants with high-priority conflicts

high_priority_conflict_rids = set(
    ptdemog_nearest.loc[
        ptdemog_nearest["HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT"].fillna(False),
        "COHORT_RID",
    ].astype(int)
)

print(
    "Participants with high-priority demographic conflicts:",
    len(high_priority_conflict_rids),
)

conflict_columns = [
    "COHORT_RID",
    "COHORT_PTID",
    "COHORT_LABEL",
    "BASELINE_DATE",
    "VISDATE_PARSED",
    "VISCODE2_CLEAN",
    "DAYS_FROM_BASELINE",
    "ABS_DAYS_FROM_BASELINE",
    "PTGENDER_CLEAN",
    "PTGENDER_LABEL",
    "PTHAND_CLEAN",
    "PTHAND_LABEL",
    "PTPLANG_CLEAN",
    "PTPLANG_LABEL",
    "PTGENDER_CLEAN_CONFLICT",
    "PTHAND_CLEAN_CONFLICT",
    "PTPLANG_CLEAN_CONFLICT",
    "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT",
    "DEMOGRAPHIC_NON_MISSING_COUNT",
    "HAS_QC_ERROR_CLEAN",
]

display(
    ptdemog_cohort.loc[
        ptdemog_cohort["COHORT_RID"].isin(high_priority_conflict_rids),
        conflict_columns,
    ].sort_values(
        [
            "COHORT_RID",
            "VISDATE_PARSED",
        ]
    )
)

### 20.1. High-priority PTDEMOG conflict interpretation

The nearest-to-baseline records are suitable for four of the five flagged participants because their displayed sex, handedness, and primary-language values remain consistent across visits.

One participant, RID 4607, has a genuine sex inconsistency: the screening record reports female, while a much later record reports male. The screening value is much closer to the clinical baseline, but the conflict should not be resolved automatically.

The participant-level high-priority flag can also be triggered by conflict fields not included in the previous display, such as birth year. All underlying conflict indicators will therefore be inspected before finalising the PTDEMOG record.

In [ ]:
# Inspect every demographic conflict flag for the five flagged participants

all_conflict_columns = [
    column
    for column in ptdemog_cohort.columns
    if column.endswith("_CONFLICT")
]

print("Available demographic conflict columns")
print("--------------------------------------")
print(all_conflict_columns)

conflict_flag_summary = (
    ptdemog_cohort.loc[
        ptdemog_cohort["COHORT_RID"].isin(high_priority_conflict_rids),
        ["COHORT_RID", "COHORT_PTID"] + all_conflict_columns,
    ]
    .groupby(
        ["COHORT_RID", "COHORT_PTID"],
        as_index=False,
    )
    .agg({
        column: lambda values: values.fillna(False).astype(bool).any()
        for column in all_conflict_columns
    })
)

display(conflict_flag_summary)

print("\nOnly conflict indicators evaluating to True")
print("-------------------------------------------")

for _, row in conflict_flag_summary.iterrows():
    true_conflicts = [
        column
        for column in all_conflict_columns
        if bool(row[column])
    ]

    print(
        f"RID {int(row['COHORT_RID'])} "
        f"({row['COHORT_PTID']}): "
        f"{true_conflicts if true_conflicts else 'None'}"
    )

### 20.2. PTDEMOG conflict decision

Four participants have conflicting cleaned birth-year values across visits, while one participant has a conflicting sex value.

For the participant-level PTDEMOG record:

- the nearest record to clinical baseline will remain the selected source row;
- conflicting birth year will not be used directly as a modelling feature;
- age should instead be derived from the selected baseline date and the cleaned birth year only when the value is consistent;
- for RID 4607, sex will be set to missing rather than choosing between the conflicting female and male entries;
- the participant will remain in the cohort because the conflict affects only one demographic feature, not cohort eligibility.

# 21. Finalise the participant-level PTDEMOG cohort

The nearest dated PTDEMOG record will be retained for each participant.

The final PTDEMOG modality will:

- preserve all 2,199 cohort participants;
- keep stable demographic features only;
- exclude disease-history fields that could encode progression;
- derive age at clinical baseline where birth year is consistent;
- set conflicting high-priority values to missing;
- retain timing and conflict indicators for audit.

In [ ]:
# Finalise one PTDEMOG record per participant

ptdemog_final = ptdemog_nearest.copy()

# Derive age at clinical baseline only when birth year is consistent
ptdemog_final["AGE_AT_BASELINE"] = (
    ptdemog_final["BASELINE_DATE"].dt.year
    - ptdemog_final["BIRTH_YEAR_CLEAN"]
)

ptdemog_final.loc[
    ptdemog_final["BIRTH_YEAR_CLEAN_CONFLICT"].fillna(False),
    ["BIRTH_YEAR_CLEAN", "AGE_AT_BASELINE"],
] = pd.NA

# Do not force a sex value for the participant with a documented conflict
ptdemog_final.loc[
    ptdemog_final["PTGENDER_CLEAN_CONFLICT"].fillna(False),
    ["PTGENDER_CLEAN", "PTGENDER_LABEL"],
] = pd.NA

ptdemog_feature_columns = [
    "COHORT_RID",
    "COHORT_PTID",
    "COHORT_LABEL",
    "BASELINE_PHASE",
    "BASELINE_DATE",
    "VISDATE_PARSED",
    "VISCODE2_CLEAN",
    "DAYS_FROM_BASELINE",
    "ABS_DAYS_FROM_BASELINE",
    "AGE_AT_BASELINE",
    "PTGENDER_CLEAN",
    "PTGENDER_LABEL",
    "PTEDUCAT_CLEAN",
    "PTHAND_CLEAN",
    "PTHAND_LABEL",
    "PTMARRY_CLEAN",
    "PTMARRY_LABEL",
    "PTPLANG_CLEAN",
    "PTPLANG_LABEL",
    "PTTLANG_CLEAN",
    "PTTLANG_LABEL",
    "PTETHCAT_CLEAN",
    "PTETHCAT_LABEL",
    "PTRACCAT_CLEAN",
    "PTRACCAT_LABEL",
    "PTNLANG_CLEAN",
    "PTNLANG_LABEL",
    "ANY_STABLE_DEMOGRAPHIC_CONFLICT",
    "HIGH_PRIORITY_DEMOGRAPHIC_CONFLICT",
    "HAS_QC_ERROR_CLEAN",
]

ptdemog_final = (
    ptdemog_final[ptdemog_feature_columns]
    .rename(
        columns={
            "COHORT_RID": "RID",
            "COHORT_PTID": "PTID",
            "VISDATE_PARSED": "PTDEMOG_DATE",
            "VISCODE2_CLEAN": "PTDEMOG_VISCODE2",
        }
    )
    .sort_values("RID")
    .reset_index(drop=True)
)

print("Final PTDEMOG cohort")
print("--------------------")
print(f"Rows: {len(ptdemog_final):,}")
print(f"Unique RID: {ptdemog_final['RID'].nunique():,}")
print(f"Duplicate RID: {ptdemog_final['RID'].duplicated().sum():,}")
print(f"Missing age at baseline: {ptdemog_final['AGE_AT_BASELINE'].isna().sum():,}")
print(f"Missing sex: {ptdemog_final['PTGENDER_CLEAN'].isna().sum():,}")

print("\nCohort distribution:")
print(
    ptdemog_final["COHORT_LABEL"]
    .value_counts(dropna=False)
)

display(ptdemog_final.head())

# 22. Save the final PTDEMOG baseline cohort

The participant-level PTDEMOG table is complete.

It contains one baseline-aligned demographic record for each of the 2,199 participants in the authoritative clinical cohort. Stable demographic variables are retained, disease-history variables are excluded, and unresolved birth-year or sex conflicts are represented as missing values rather than forced assignments.

The final table will now be saved as the authoritative PTDEMOG modality manifest.

In [ ]:
# Save and verify the final PTDEMOG participant-level cohort

PTDEMOG_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "ptdemog"
)

PTDEMOG_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PTDEMOG_FINAL_PATH = (
    PTDEMOG_OUTPUT_DIR
    / "ptdemog_baseline_aligned_cohort.csv"
)

ptdemog_final.to_csv(
    PTDEMOG_FINAL_PATH,
    index=False,
)

ptdemog_final_reloaded = pd.read_csv(
    PTDEMOG_FINAL_PATH,
    low_memory=False,
)

print("Final PTDEMOG file")
print("------------------")
print("Path:", PTDEMOG_FINAL_PATH)
print("Exists:", PTDEMOG_FINAL_PATH.exists())
print(f"Rows: {len(ptdemog_final_reloaded):,}")
print(f"Columns: {len(ptdemog_final_reloaded.columns)}")
print(
    "Unique RID:",
    f"{ptdemog_final_reloaded['RID'].nunique():,}"
)
print(
    "Duplicate RID:",
    f"{ptdemog_final_reloaded['RID'].duplicated().sum():,}"
)

# 23. Align MMSE with the authoritative clinical cohort

The cleaned longitudinal MMSE table will now be matched to the authoritative four-group clinical cohort.

Unlike PTDEMOG, MMSE is time-varying and can reflect disease progression. Therefore, only measurements sufficiently close to the DXSUM clinical baseline should be considered.

This step will first load the cleaned MMSE table and inspect its identifiers, visit dates, score variables, and quality-control fields before applying any timing window.

In [ ]:
# Load the processed MMSE feature tables

MMSE_DIR = (
    PROCESSED_DIR
    / "mmse"
)

MMSE_FILES = {
    "full_features": (
        MMSE_DIR
        / "mmse_longitudinal_features.csv"
    ),
    "primary_features": (
        MMSE_DIR
        / "mmse_longitudinal_primary_features_v2.csv"
    ),
    "optional_domain_features": (
        MMSE_DIR
        / "mmse_longitudinal_optional_domain_features_v2.csv"
    ),
}

mmse_tables = {}

for name, path in MMSE_FILES.items():
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(
            f"Required MMSE file not found: {path}"
        )

    mmse_tables[name] = pd.read_csv(
        path,
        low_memory=False,
    )

    print(f"Rows: {len(mmse_tables[name]):,}")
    print(f"Columns: {len(mmse_tables[name].columns)}")
    print(mmse_tables[name].columns.tolist())

# 24. Audit MMSE coverage and timing relative to clinical baseline

The three MMSE files contain the same 14,804 longitudinal assessments, but with different feature sets:

- `primary_features` contains the main MMSE total score intended for modelling;
- `optional_domain_features` contains reconstructed MMSE domain scores;
- `full_features` contains the detailed audit and quality-control fields.

The existing `MMSE_TEMPORALLY_ELIGIBLE` flag will not be assumed to match the newly reconstructed authoritative cohort. MMSE timing will be recalculated directly against each participant’s DXSUM baseline date.

This step will use the primary table for participant coverage and merge selected quality-control fields from the full table through `MMSE_RECORD_KEY`.

In [ ]:
# Prepare the MMSE assessment table and audit cohort coverage

mmse_primary = mmse_tables["primary_features"].copy()
mmse_full = mmse_tables["full_features"].copy()

# Standardise identifiers and dates
mmse_primary["RID"] = pd.to_numeric(
    mmse_primary["RID"],
    errors="coerce"
).astype("Int64")

mmse_full["RID"] = pd.to_numeric(
    mmse_full["RID"],
    errors="coerce"
).astype("Int64")

mmse_primary["VISDATE"] = pd.to_datetime(
    mmse_primary["VISDATE"],
    errors="coerce"
)

mmse_full["VISDATE"] = pd.to_datetime(
    mmse_full["VISDATE"],
    errors="coerce"
)

# Attach audit fields from the full MMSE table
mmse_qc_columns = [
    "MMSE_RECORD_KEY",
    "MMSE_EXPLICITLY_NOT_DONE",
    "MMSE_REMOTE_ASSESSMENT",
    "MMSE_HAS_QC_ERROR",
    "MMSE_TOTAL_MISMATCH",
    "MMSE_DUPLICATE_ASSESSMENT_KEY",
    "MMSE_CONFLICTING_DUPLICATE_SCORE",
]

mmse_primary_audited = mmse_primary.merge(
    mmse_full[mmse_qc_columns],
    on="MMSE_RECORD_KEY",
    how="left",
    validate="one_to_one",
)

# Merge with the authoritative clinical cohort
mmse_cohort = mmse_primary_audited.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "FIRST_AD_DATE",
        ]
    ].rename(columns={"PTID": "COHORT_PTID"}),
    on="RID",
    how="inner",
    validate="many_to_one",
)

mmse_cohort["BASELINE_DATE"] = pd.to_datetime(
    mmse_cohort["BASELINE_DATE"],
    errors="coerce"
)

mmse_cohort["FIRST_AD_DATE"] = pd.to_datetime(
    mmse_cohort["FIRST_AD_DATE"],
    errors="coerce"
)

mmse_cohort["DAYS_FROM_BASELINE"] = (
    mmse_cohort["VISDATE"] - mmse_cohort["BASELINE_DATE"]
).dt.days

mmse_cohort["ABS_DAYS_FROM_BASELINE"] = (
    mmse_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

mmse_rids = set(
    mmse_primary["RID"]
    .dropna()
    .astype(int)
)

matched_rids = cohort_rids & mmse_rids

print("MMSE cohort coverage")
print("--------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in MMSE: {len(matched_rids):,}")
print(f"Participants absent from MMSE: {len(cohort_rids - mmse_rids):,}")

print("\nMatched MMSE records")
print("--------------------")
print(f"Rows: {len(mmse_cohort):,}")
print(f"Unique RID: {mmse_cohort['RID'].nunique():,}")
print(f"Missing VISDATE: {mmse_cohort['VISDATE'].isna().sum():,}")
print(
    "Usable MMSE scores:",
    f"{mmse_cohort['MMSE_SCORE_USABLE'].fillna(False).sum():,}"
)

print("\nCoverage by cohort label")
print("------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_MMSE=authoritative_clinical_cohort["RID"].isin(matched_rids)
    )
    .groupby("COHORT_LABEL")["HAS_MMSE"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nMMSE quality-control flags")
print("--------------------------")
for column in [
    "MMSE_EXPLICITLY_NOT_DONE",
    "MMSE_REMOTE_ASSESSMENT",
    "MMSE_HAS_QC_ERROR",
    "MMSE_TOTAL_MISMATCH",
    "MMSE_DUPLICATE_ASSESSMENT_KEY",
    "MMSE_CONFLICTING_DUPLICATE_SCORE",
]:
    print(
        f"{column}: "
        f"{mmse_cohort[column].fillna(False).astype(bool).sum():,}"
    )

### 24.1. MMSE coverage and quality-control summary

MMSE is available for all 2,199 participants in the authoritative cohort.

Among 9,833 matched longitudinal records:

- 9,709 have a usable MMSE score;
- 81 assessments were explicitly not completed;
- 6 were remote assessments;
- no records have a formal QC error;
- 91 have a mismatch between the reported and reconstructed MMSE total;
- no duplicate assessment keys or conflicting duplicate scores were found.

The existing `MMSE_SCORE_USABLE` flag will be respected. The next step is to identify the nearest usable MMSE assessment to each participant’s clinical baseline and inspect how many fall within the intended ±90-day window.

In [ ]:
# Identify the nearest usable MMSE assessment to clinical baseline

mmse_eligible = mmse_cohort.loc[
    mmse_cohort["MMSE_SCORE_USABLE"].fillna(False)
    & mmse_cohort["VISDATE"].notna()
    & mmse_cohort["MMSE_TOTAL_SCORE"].notna()
].copy()

# Ensure pMCI measurements do not occur on or after confirmed conversion
mmse_eligible["IS_BEFORE_PMCICONVERSION"] = (
    ~mmse_eligible["COHORT_LABEL"].eq("pMCI")
    | mmse_eligible["FIRST_AD_DATE"].isna()
    | (mmse_eligible["VISDATE"] < mmse_eligible["FIRST_AD_DATE"])
)

mmse_eligible = mmse_eligible.loc[
    mmse_eligible["IS_BEFORE_PMCICONVERSION"]
].copy()

mmse_nearest = (
    mmse_eligible
    .sort_values(
        [
            "RID",
            "ABS_DAYS_FROM_BASELINE",
            "DAYS_FROM_BASELINE",
            "VISDATE",
        ],
        ascending=[True, True, True, True],
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .copy()
)

mmse_nearest["TIMING_DIRECTION"] = pd.cut(
    mmse_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

print("Nearest usable MMSE assessment")
print("------------------------------")
print(f"Selected participants: {len(mmse_nearest):,}")
print(f"Unique RID: {mmse_nearest['RID'].nunique():,}")
print(
    "Cohort participants without a usable dated MMSE:",
    f"{len(cohort_rids - set(mmse_nearest['RID'].astype(int))):,}"
)

print("\nTiming direction")
print("----------------")
print(
    mmse_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [30, 60, 90, 180, 365]:
    count = mmse_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(authoritative_clinical_cohort)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}% of cohort)"
    )

print("\nNearest MMSE timing summary")
print("---------------------------")
print(
    mmse_nearest["ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nCoverage within ±90 days by cohort label")
print("-----------------------------------------")
display(
    authoritative_clinical_cohort[
        ["RID", "COHORT_LABEL"]
    ]
    .assign(
        HAS_MMSE_WITHIN_90D=lambda df: df["RID"].isin(
            set(
                mmse_nearest.loc[
                    mmse_nearest["ABS_DAYS_FROM_BASELINE"] <= 90,
                    "RID",
                ]
            )
        )
    )
    .groupby("COHORT_LABEL")["HAS_MMSE_WITHIN_90D"]
    .value_counts()
    .unstack(fill_value=0)
)

### 24.2. MMSE baseline-alignment summary

A usable dated MMSE assessment is available for 2,198 of the 2,199 cohort participants.

Using the nearest eligible assessment:

- 1,828 participants have MMSE within ±90 days of clinical baseline;
- 2,106 are within ±180 days;
- 54 selected assessments occur after baseline;
- one participant has no usable dated MMSE assessment.

Because MMSE is time-varying, the 370 participants outside the ±90-day window should not automatically be treated as having baseline MMSE. Before finalising the modality, the out-of-window records and the single missing participant must be inspected.

In [ ]:
# Inspect MMSE records outside the ±90-day baseline window

mmse_outside_90d = mmse_nearest.loc[
    mmse_nearest["ABS_DAYS_FROM_BASELINE"] > 90
].copy()

missing_mmse_rids = (
    cohort_rids
    - set(mmse_nearest["RID"].dropna().astype(int))
)

print("MMSE records outside ±90 days")
print("------------------------------")
print(f"Participants: {len(mmse_outside_90d):,}")

print("\nTiming direction outside ±90 days")
print("---------------------------------")
print(
    mmse_outside_90d["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance categories")
print("-------------------")
print(
    pd.cut(
        mmse_outside_90d["ABS_DAYS_FROM_BASELINE"],
        bins=[90, 180, 365, 730, float("inf")],
        labels=[
            "91–180 days",
            "181–365 days",
            "366–730 days",
            ">730 days",
        ],
        include_lowest=True,
    ).value_counts(sort=False)
)

print("\nOut-of-window participants by cohort label")
print("------------------------------------------")
print(
    mmse_outside_90d["COHORT_LABEL"]
    .value_counts(dropna=False)
)

print("\nPost-baseline MMSE records outside ±90 days")
print("-------------------------------------------")
display(
    mmse_outside_90d.loc[
        mmse_outside_90d["DAYS_FROM_BASELINE"] > 90,
        [
            "RID",
            "COHORT_PTID",
            "COHORT_LABEL",
            "BASELINE_DATE",
            "VISDATE",
            "VISCODE2",
            "DAYS_FROM_BASELINE",
            "ABS_DAYS_FROM_BASELINE",
            "MMSE_TOTAL_SCORE",
            "FIRST_AD_DATE",
            "MMSE_REMOTE_ASSESSMENT",
            "MMSE_TOTAL_MISMATCH",
        ],
    ].sort_values(
        "DAYS_FROM_BASELINE",
        ascending=False,
    )
)

print("\nParticipant without a usable dated MMSE")
print("---------------------------------------")
display(
    authoritative_clinical_cohort.loc[
        authoritative_clinical_cohort["RID"].isin(missing_mmse_rids),
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ],
    ]
)

### 24.3. MMSE timing decision

MMSE will be treated as available at baseline only when the nearest usable assessment falls within ±90 days of the DXSUM clinical baseline.

The 370 out-of-window measurements will not be used as baseline predictors. They will be represented as missing MMSE rather than causing participant exclusion. This is especially important for post-baseline measurements, which may already reflect clinical progression.

The final MMSE manifest will therefore retain all 2,199 cohort participants and include:

- the nearest eligible MMSE total score within ±90 days;
- optional domain scores from the same assessment;
- the signed and absolute timing difference;
- an MMSE availability indicator;
- quality-control flags from the selected record.

In [ ]:
# Construct the baseline-aligned MMSE modality table

mmse_optional = mmse_tables["optional_domain_features"].copy()

mmse_optional["RID"] = pd.to_numeric(
    mmse_optional["RID"],
    errors="coerce"
).astype("Int64")

mmse_optional["VISDATE"] = pd.to_datetime(
    mmse_optional["VISDATE"],
    errors="coerce"
)

optional_columns = [
    "MMSE_RECORD_KEY",
    "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE",
    "MMSE_DOMAIN_TOTAL_CONSISTENT",
]

# Retain only nearest usable records within the ±90-day window
mmse_selected_90d = (
    mmse_nearest.loc[
        mmse_nearest["ABS_DAYS_FROM_BASELINE"] <= 90
    ]
    .merge(
        mmse_optional[optional_columns],
        on="MMSE_RECORD_KEY",
        how="left",
        validate="one_to_one",
    )
    .copy()
)

mmse_selected_columns = [
    "RID",
    "MMSE_RECORD_KEY",
    "VISDATE",
    "VISCODE2",
    "MMSE_VISIT_CODE",
    "MMSE_TOTAL_SCORE",
    "MMSE_ORIENTATION_SCORE",
    "MMSE_REGISTRATION_SCORE",
    "MMSE_ATTENTION_SCORE",
    "MMSE_DELAYED_RECALL_SCORE",
    "MMSE_LANGUAGE_COMMAND_SCORE",
    "MMSE_DOMAIN_TOTAL_CONSISTENT",
    "DAYS_FROM_BASELINE",
    "ABS_DAYS_FROM_BASELINE",
    "MMSE_REMOTE_ASSESSMENT",
    "MMSE_TOTAL_MISMATCH",
    "MMSE_HAS_QC_ERROR",
]

mmse_baseline_aligned = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        mmse_selected_90d[mmse_selected_columns],
        on="RID",
        how="left",
        validate="one_to_one",
    )
)

mmse_baseline_aligned["MMSE_AVAILABLE"] = (
    mmse_baseline_aligned["MMSE_TOTAL_SCORE"].notna()
)

mmse_baseline_aligned = mmse_baseline_aligned.rename(
    columns={
        "VISDATE": "MMSE_DATE",
        "VISCODE2": "MMSE_VISCODE2",
    }
)

print("Baseline-aligned MMSE cohort")
print("----------------------------")
print(f"Rows: {len(mmse_baseline_aligned):,}")
print(f"Unique RID: {mmse_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{mmse_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "MMSE available within ±90 days:",
    f"{mmse_baseline_aligned['MMSE_AVAILABLE'].sum():,}"
)
print(
    "MMSE unavailable:",
    f"{(~mmse_baseline_aligned['MMSE_AVAILABLE']).sum():,}"
)

print("\nAvailability by cohort label:")
display(
    pd.crosstab(
        mmse_baseline_aligned["COHORT_LABEL"],
        mmse_baseline_aligned["MMSE_AVAILABLE"],
        margins=True,
    )
)

### 24.4. MMSE baseline-alignment decision

The final MMSE modality contains all 2,199 cohort participants.

A usable MMSE assessment within ±90 days of clinical baseline is available for 1,828 participants. For the remaining 371 participants, MMSE will be marked as unavailable rather than causing participant exclusion.

This preserves the full multimodal cohort while preventing temporally distant MMSE measurements from being used as baseline predictors.

# 25. Save the final baseline-aligned MMSE cohort

In [ ]:
MMSE_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "mmse"
)

MMSE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MMSE_FINAL_PATH = (
    MMSE_OUTPUT_DIR
    / "mmse_baseline_aligned_cohort.csv"
)

mmse_baseline_aligned.to_csv(
    MMSE_FINAL_PATH,
    index=False,
)

mmse_reloaded = pd.read_csv(
    MMSE_FINAL_PATH,
    low_memory=False,
)

print("Final MMSE file")
print("---------------")
print("Path:", MMSE_FINAL_PATH)
print("Exists:", MMSE_FINAL_PATH.exists())
print(f"Rows: {len(mmse_reloaded):,}")
print(f"Unique RID: {mmse_reloaded['RID'].nunique():,}")
print(f"Duplicate RID: {mmse_reloaded['RID'].duplicated().sum():,}")
print(
    "MMSE available:",
    f"{mmse_reloaded['MMSE_AVAILABLE'].sum():,}"
)

# 26. Load the processed FAQ table

The final dated FAQ modelling table is stored directly in the processed folder:

`/content/drive/MyDrive/adni_mri/adni_non_imaging/processed/faq_model_ready_dated_longitudinal.csv`

The separate `faq_clean_longitudinal_interim.csv` file is a broader intermediate cleaning output. For baseline cohort construction, the model-ready dated file will be used.

In [ ]:
# Load the processed longitudinal FAQ modelling table

FAQ_PATH = (
    PROCESSED_DIR
    / "faq_model_ready_dated_longitudinal.csv"
)

print("FAQ path:")
print(FAQ_PATH)
print("Exists:", FAQ_PATH.exists())

if not FAQ_PATH.exists():
    raise FileNotFoundError(
        f"The processed FAQ file was not found: {FAQ_PATH}"
    )

faq_cleaned = pd.read_csv(
    FAQ_PATH,
    low_memory=False,
)

print(f"\nFAQ rows: {len(faq_cleaned):,}")
print(f"FAQ columns: {len(faq_cleaned.columns)}")

print("\nColumns:")
print(faq_cleaned.columns.tolist())

display(faq_cleaned.head())

# 27. Audit FAQ coverage and timing relative to clinical baseline

The model-ready FAQ table contains dated longitudinal `FAQTOTAL` measurements.

FAQ is time-varying and can reflect functional decline, so timing will be recalculated against the authoritative DXSUM baseline rather than relying only on visit codes.

This step will:

- standardise identifiers and dates;
- merge FAQ with the authoritative cohort;
- calculate the distance from clinical baseline;
- inspect participant coverage, missing scores, and repeated-record flags.

In [ ]:
# Prepare FAQ and audit cohort coverage

faq_cleaned["RID"] = pd.to_numeric(
    faq_cleaned["RID"],
    errors="coerce",
).astype("Int64")

faq_cleaned["VISDATE"] = pd.to_datetime(
    faq_cleaned["VISDATE"],
    errors="coerce",
)

faq_cleaned["FAQTOTAL"] = pd.to_numeric(
    faq_cleaned["FAQTOTAL"],
    errors="coerce",
)

faq_cohort = faq_cleaned.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "FIRST_AD_DATE",
        ]
    ].rename(columns={"PTID": "COHORT_PTID"}),
    on="RID",
    how="inner",
    validate="many_to_one",
)

faq_cohort["BASELINE_DATE"] = pd.to_datetime(
    faq_cohort["BASELINE_DATE"],
    errors="coerce",
)

faq_cohort["FIRST_AD_DATE"] = pd.to_datetime(
    faq_cohort["FIRST_AD_DATE"],
    errors="coerce",
)

faq_cohort["DAYS_FROM_BASELINE"] = (
    faq_cohort["VISDATE"] - faq_cohort["BASELINE_DATE"]
).dt.days

faq_cohort["ABS_DAYS_FROM_BASELINE"] = (
    faq_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

faq_rids = set(
    faq_cleaned["RID"]
    .dropna()
    .astype(int)
)

matched_faq_rids = cohort_rids & faq_rids

print("FAQ cohort coverage")
print("-------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in FAQ: {len(matched_faq_rids):,}")
print(f"Participants absent from FAQ: {len(cohort_rids - faq_rids):,}")

print("\nMatched FAQ records")
print("-------------------")
print(f"Rows: {len(faq_cohort):,}")
print(f"Unique RID: {faq_cohort['RID'].nunique():,}")
print(f"Missing VISDATE: {faq_cohort['VISDATE'].isna().sum():,}")
print(f"Missing FAQTOTAL: {faq_cohort['FAQTOTAL'].isna().sum():,}")
print(
    "FAQTOTAL outside 0–30:",
    (
        faq_cohort["FAQTOTAL"].notna()
        & ~faq_cohort["FAQTOTAL"].between(0, 30)
    ).sum(),
)

print("\nRepeated-record flags")
print("---------------------")
print(
    "Repeated RID–VISCODE2:",
    faq_cohort["flag_repeated_rid_viscode2"]
    .fillna(False)
    .astype(bool)
    .sum(),
)
print(
    "Repeated RID–VISDATE:",
    faq_cohort["flag_repeated_rid_visdate"]
    .fillna(False)
    .astype(bool)
    .sum(),
)

print("\nCoverage by cohort label")
print("------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_FAQ=authoritative_clinical_cohort["RID"].isin(matched_faq_rids)
    )
    .groupby("COHORT_LABEL")["HAS_FAQ"]
    .value_counts()
    .unstack(fill_value=0)
)

### 27.1. FAQ coverage and duplicate-record summary

FAQ coverage is very high:

- 2,189 of 2,199 participants are present in the processed FAQ table;
- all matched records have valid dates and valid `FAQTOTAL` values between 0 and 30;
- 10 participants are absent from FAQ entirely;
- a small number of records are flagged as repeated by visit code or visit date.

Before selecting the nearest FAQ assessment to baseline, the repeated records must be inspected to determine whether they are exact duplicates or conflicting assessments.

In [ ]:
# Inspect FAQ records flagged as repeated

faq_repeated = faq_cohort.loc[
    faq_cohort["flag_repeated_rid_viscode2"].fillna(False)
    | faq_cohort["flag_repeated_rid_visdate"].fillna(False)
].copy()

print("Flagged repeated FAQ records")
print("----------------------------")
print(f"Rows: {len(faq_repeated):,}")
print(f"Participants: {faq_repeated['RID'].nunique():,}")

display(
    faq_repeated[
        [
            "RID",
            "COHORT_PTID",
            "COHORT_LABEL",
            "PHASE",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "FAQTOTAL",
            "SOURCE",
            "ID",
            "SITEID",
            "DAYS_FROM_BASELINE",
            "ABS_DAYS_FROM_BASELINE",
            "flag_repeated_rid_viscode2",
            "flag_repeated_rid_visdate",
        ]
    ].sort_values(
        ["RID", "VISDATE", "VISCODE2", "ID"]
    )
)

print("\nRepeated-date score consistency")
print("--------------------------------")

faq_repeated_date_summary = (
    faq_repeated.loc[
        faq_repeated["flag_repeated_rid_visdate"].fillna(False)
    ]
    .groupby(["RID", "VISDATE"], as_index=False)
    .agg(
        RECORD_COUNT=("FAQTOTAL", "size"),
        UNIQUE_FAQTOTAL=("FAQTOTAL", "nunique"),
        MIN_FAQTOTAL=("FAQTOTAL", "min"),
        MAX_FAQTOTAL=("FAQTOTAL", "max"),
    )
)

display(faq_repeated_date_summary)

### 27.2. FAQ repeated-record decision

The repeated FAQ records do not affect baseline selection:

- five participants have two records on the same date;
- four of those pairs have identical `FAQTOTAL` values;
- RID 91 has scores of 14 and 15 on the same date, but that date is 741 days after baseline;
- RID 6452 has a repeated visit code on two different dates, both more than seven years after baseline.

All flagged records are far outside the intended ±90-day baseline window. They can therefore remain in the longitudinal source table without requiring manual score resolution.

The next step is to select the nearest valid FAQ assessment to baseline, while ensuring that pMCI measurements occur before conversion.

In [ ]:
# Select the nearest valid FAQ assessment to clinical baseline

faq_eligible = faq_cohort.loc[
    faq_cohort["VISDATE"].notna()
    & faq_cohort["FAQTOTAL"].between(0, 30)
].copy()

# Prevent post-conversion measurements from being selected for pMCI
faq_eligible["IS_BEFORE_PMCICONVERSION"] = (
    ~faq_eligible["COHORT_LABEL"].eq("pMCI")
    | faq_eligible["FIRST_AD_DATE"].isna()
    | (faq_eligible["VISDATE"] < faq_eligible["FIRST_AD_DATE"])
)

faq_eligible = faq_eligible.loc[
    faq_eligible["IS_BEFORE_PMCICONVERSION"]
].copy()

# Prefer the closest assessment.
# For an equal absolute distance, prefer the pre-baseline record.
faq_eligible["POST_BASELINE_TIEBREAKER"] = (
    faq_eligible["DAYS_FROM_BASELINE"] > 0
).astype(int)

faq_nearest = (
    faq_eligible
    .sort_values(
        [
            "RID",
            "ABS_DAYS_FROM_BASELINE",
            "POST_BASELINE_TIEBREAKER",
            "VISDATE",
            "ID",
        ],
        ascending=[True, True, True, True, True],
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .copy()
)

faq_nearest["TIMING_DIRECTION"] = pd.cut(
    faq_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

selected_faq_rids = set(
    faq_nearest["RID"]
    .dropna()
    .astype(int)
)

print("Nearest valid FAQ assessment")
print("----------------------------")
print(f"Selected participants: {len(faq_nearest):,}")
print(f"Unique RID: {faq_nearest['RID'].nunique():,}")
print(
    "Cohort participants without a valid dated FAQ:",
    f"{len(cohort_rids - selected_faq_rids):,}"
)

print("\nTiming direction")
print("----------------")
print(
    faq_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [30, 60, 90, 180, 365]:
    count = faq_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(authoritative_clinical_cohort)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}% of cohort)"
    )

print("\nNearest FAQ timing summary")
print("--------------------------")
print(
    faq_nearest["ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nCoverage within ±90 days by cohort label")
print("-----------------------------------------")
display(
    authoritative_clinical_cohort[
        ["RID", "COHORT_LABEL"]
    ]
    .assign(
        HAS_FAQ_WITHIN_90D=lambda df: df["RID"].isin(
            set(
                faq_nearest.loc[
                    faq_nearest["ABS_DAYS_FROM_BASELINE"] <= 90,
                    "RID",
                ]
            )
        )
    )
    .groupby("COHORT_LABEL")["HAS_FAQ_WITHIN_90D"]
    .value_counts()
    .unstack(fill_value=0)
)

### 27.3. FAQ baseline-alignment decision

FAQ will be treated as available at baseline when the nearest valid assessment falls within ±90 days of the authoritative DXSUM baseline.

This gives baseline FAQ coverage for 2,098 of the 2,199 participants. The remaining 101 participants will retain their cohort membership but have FAQ marked as unavailable.

The selected FAQ record is always dated, has a valid `FAQTOTAL` between 0 and 30, and for pMCI participants occurs before conversion to Alzheimer’s disease.

# 28. Construct and save the final baseline-aligned FAQ cohort

In [ ]:
# Construct the participant-level baseline-aligned FAQ table

faq_selected_90d = faq_nearest.loc[
    faq_nearest["ABS_DAYS_FROM_BASELINE"] <= 90,
    [
        "RID",
        "PHASE",
        "VISCODE",
        "VISCODE2",
        "VISDATE",
        "FAQTOTAL",
        "SOURCE",
        "ID",
        "SITEID",
        "DAYS_FROM_BASELINE",
        "ABS_DAYS_FROM_BASELINE",
        "flag_repeated_rid_viscode2",
        "flag_repeated_rid_visdate",
    ],
].copy()

faq_baseline_aligned = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        faq_selected_90d,
        on="RID",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "PHASE": "FAQ_PHASE",
            "VISCODE": "FAQ_VISCODE",
            "VISCODE2": "FAQ_VISCODE2",
            "VISDATE": "FAQ_DATE",
            "SOURCE": "FAQ_SOURCE",
            "ID": "FAQ_ID",
            "SITEID": "FAQ_SITEID",
        }
    )
)

faq_baseline_aligned["FAQ_AVAILABLE"] = (
    faq_baseline_aligned["FAQTOTAL"].notna()
)

print("Baseline-aligned FAQ cohort")
print("---------------------------")
print(f"Rows: {len(faq_baseline_aligned):,}")
print(f"Unique RID: {faq_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{faq_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "FAQ available within ±90 days:",
    f"{faq_baseline_aligned['FAQ_AVAILABLE'].sum():,}"
)
print(
    "FAQ unavailable:",
    f"{(~faq_baseline_aligned['FAQ_AVAILABLE']).sum():,}"
)

FAQ_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "faq"
)

FAQ_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FAQ_FINAL_PATH = (
    FAQ_OUTPUT_DIR
    / "faq_baseline_aligned_cohort.csv"
)

faq_baseline_aligned.to_csv(
    FAQ_FINAL_PATH,
    index=False,
)

faq_reloaded = pd.read_csv(
    FAQ_FINAL_PATH,
    low_memory=False,
)

print("\nSaved FAQ file")
print("--------------")
print("Path:", FAQ_FINAL_PATH)
print("Exists:", FAQ_FINAL_PATH.exists())
print(f"Rows after reload: {len(faq_reloaded):,}")
print(f"Unique RID after reload: {faq_reloaded['RID'].nunique():,}")
print(f"Duplicate RID after reload: {faq_reloaded['RID'].duplicated().sum():,}")

# 29. Load the cleaned longitudinal ADAS table

The available ADAS output is:

`/content/drive/MyDrive/adni_mri/adni_non_imaging/processed/adas_clean_longitudinal_interim.csv`

Although the filename contains `interim`, it can be used as the cleaned longitudinal source for constructing the final participant-level baseline-aligned ADAS table.

In [ ]:
# Load the cleaned longitudinal ADAS table

ADAS_PATH = (
    PROCESSED_DIR
    / "adas_clean_longitudinal_interim.csv"
)

print("ADAS path:")
print(ADAS_PATH)
print("Exists:", ADAS_PATH.exists())

if not ADAS_PATH.exists():
    raise FileNotFoundError(
        f"The cleaned ADAS file was not found: {ADAS_PATH}"
    )

adas_cleaned = pd.read_csv(
    ADAS_PATH,
    low_memory=False,
)

print(f"\nADAS rows: {len(adas_cleaned):,}")
print(f"ADAS columns: {len(adas_cleaned.columns)}")

print("\nColumns:")
print(adas_cleaned.columns.tolist())

display(adas_cleaned.head())

# 30. Audit ADAS coverage, score availability, and quality-control flags

The cleaned ADAS table contains both:

- `TOTSCORE`, representing the traditional ADAS-Cog total;
- `TOTAL13`, representing the extended 13-item ADAS-Cog total.

Before selecting baseline-aligned records, the table must be checked for:

- participant coverage;
- missing dates and totals;
- provisional exclusions;
- unresolved QC errors;
- invalid score ranges;
- availability of `TOTSCORE` and `TOTAL13` by ADNI phase.

Records already marked with `flag_provisional_exclusion = True` will not be considered eligible for baseline modelling.

In [ ]:
# Prepare ADAS and audit cohort coverage and data quality

adas_cleaned["RID"] = pd.to_numeric(
    adas_cleaned["RID"],
    errors="coerce",
).astype("Int64")

adas_cleaned["VISDATE"] = pd.to_datetime(
    adas_cleaned["VISDATE"],
    errors="coerce",
)

for column in ["TOTSCORE", "TOTAL13"]:
    adas_cleaned[column] = pd.to_numeric(
        adas_cleaned[column],
        errors="coerce",
    )

adas_cohort = adas_cleaned.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "FIRST_AD_DATE",
        ]
    ].rename(columns={"PTID": "COHORT_PTID"}),
    on="RID",
    how="inner",
    validate="many_to_one",
)

adas_cohort["BASELINE_DATE"] = pd.to_datetime(
    adas_cohort["BASELINE_DATE"],
    errors="coerce",
)

adas_cohort["FIRST_AD_DATE"] = pd.to_datetime(
    adas_cohort["FIRST_AD_DATE"],
    errors="coerce",
)

adas_cohort["DAYS_FROM_BASELINE"] = (
    adas_cohort["VISDATE"] - adas_cohort["BASELINE_DATE"]
).dt.days

adas_cohort["ABS_DAYS_FROM_BASELINE"] = (
    adas_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

adas_rids = set(
    adas_cleaned["RID"]
    .dropna()
    .astype(int)
)

matched_adas_rids = cohort_rids & adas_rids

print("ADAS cohort coverage")
print("--------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in ADAS: {len(matched_adas_rids):,}")
print(f"Participants absent from ADAS: {len(cohort_rids - adas_rids):,}")

print("\nMatched ADAS records")
print("--------------------")
print(f"Rows: {len(adas_cohort):,}")
print(f"Unique RID: {adas_cohort['RID'].nunique():,}")
print(f"Missing VISDATE: {adas_cohort['VISDATE'].isna().sum():,}")
print(f"Missing TOTSCORE: {adas_cohort['TOTSCORE'].isna().sum():,}")
print(f"Missing TOTAL13: {adas_cohort['TOTAL13'].isna().sum():,}")
print(
    "Both totals missing:",
    (
        adas_cohort["TOTSCORE"].isna()
        & adas_cohort["TOTAL13"].isna()
    ).sum(),
)

print("\nQuality-control flags")
print("---------------------")
for column in [
    "flag_explicitly_not_done",
    "flag_unresolved_qc_error",
    "flag_missing_both_totals",
    "flag_totscore_out_of_range",
    "flag_total13_out_of_range",
    "flag_missing_rid",
    "flag_missing_visit_date",
    "flag_provisional_exclusion",
]:
    print(
        f"{column}: "
        f"{adas_cohort[column].fillna(False).astype(bool).sum():,}"
    )

print("\nScore availability by phase")
print("---------------------------")
display(
    adas_cohort.groupby("PHASE").agg(
        RECORDS=("RID", "size"),
        UNIQUE_RID=("RID", "nunique"),
        TOTSCORE_AVAILABLE=("TOTSCORE", "count"),
        TOTAL13_AVAILABLE=("TOTAL13", "count"),
        PROVISIONALLY_EXCLUDED=(
            "flag_provisional_exclusion",
            lambda values: values.fillna(False).astype(bool).sum(),
        ),
    )
)

print("\nCoverage by cohort label")
print("------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_ADAS=authoritative_clinical_cohort["RID"].isin(
            matched_adas_rids
        )
    )
    .groupby("COHORT_LABEL")["HAS_ADAS"]
    .value_counts()
    .unstack(fill_value=0)
)

### 30.1. ADAS coverage and quality-control summary

ADAS coverage is very high:

- 2,193 of 2,199 participants are present in the cleaned ADAS table;
- all matched records have valid visit dates;
- `TOTSCORE` is available for every matched record;
- `TOTAL13` is missing in 84 records, so `TOTSCORE` should be the primary ADAS feature;
- no matched records are flagged for exclusion, unresolved QC errors, invalid score ranges, missing identifiers, or missing dates.

The next step is to select the nearest valid ADAS assessment to the authoritative clinical baseline and inspect coverage within the intended ±90-day window.

In [ ]:
# Select the nearest valid ADAS assessment to clinical baseline

adas_eligible = adas_cohort.loc[
    adas_cohort["VISDATE"].notna()
    & adas_cohort["TOTSCORE"].notna()
    & ~adas_cohort["flag_provisional_exclusion"].fillna(False)
].copy()

# Prevent post-conversion measurements from being selected for pMCI
adas_eligible["IS_BEFORE_PMCICONVERSION"] = (
    ~adas_eligible["COHORT_LABEL"].eq("pMCI")
    | adas_eligible["FIRST_AD_DATE"].isna()
    | (adas_eligible["VISDATE"] < adas_eligible["FIRST_AD_DATE"])
)

adas_eligible = adas_eligible.loc[
    adas_eligible["IS_BEFORE_PMCICONVERSION"]
].copy()

# For equal absolute distance, prefer the pre-baseline assessment
adas_eligible["POST_BASELINE_TIEBREAKER"] = (
    adas_eligible["DAYS_FROM_BASELINE"] > 0
).astype(int)

adas_nearest = (
    adas_eligible
    .sort_values(
        [
            "RID",
            "ABS_DAYS_FROM_BASELINE",
            "POST_BASELINE_TIEBREAKER",
            "VISDATE",
        ],
        ascending=[True, True, True, True],
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .copy()
)

adas_nearest["TIMING_DIRECTION"] = pd.cut(
    adas_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

selected_adas_rids = set(
    adas_nearest["RID"]
    .dropna()
    .astype(int)
)

print("Nearest valid ADAS assessment")
print("-----------------------------")
print(f"Selected participants: {len(adas_nearest):,}")
print(f"Unique RID: {adas_nearest['RID'].nunique():,}")
print(
    "Cohort participants without a valid dated ADAS:",
    f"{len(cohort_rids - selected_adas_rids):,}"
)

print("\nTiming direction")
print("----------------")
print(
    adas_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [30, 60, 90, 180, 365]:
    count = adas_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(authoritative_clinical_cohort)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}% of cohort)"
    )

print("\nNearest ADAS timing summary")
print("---------------------------")
print(
    adas_nearest["ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nCoverage within ±90 days by cohort label")
print("-----------------------------------------")
display(
    authoritative_clinical_cohort[
        ["RID", "COHORT_LABEL"]
    ]
    .assign(
        HAS_ADAS_WITHIN_90D=lambda df: df["RID"].isin(
            set(
                adas_nearest.loc[
                    adas_nearest["ABS_DAYS_FROM_BASELINE"] <= 90,
                    "RID",
                ]
            )
        )
    )
    .groupby("COHORT_LABEL")["HAS_ADAS_WITHIN_90D"]
    .value_counts()
    .unstack(fill_value=0)
)

### 30.2. ADAS baseline-alignment decision

ADAS will be considered available at baseline when the nearest valid assessment falls within ±90 days of the authoritative DXSUM baseline.

This gives baseline ADAS coverage for 2,096 of the 2,199 participants. The remaining 103 participants will remain in the cohort with ADAS marked as unavailable.

`TOTSCORE` will be retained as the primary ADAS feature because it is available for every selected record. `TOTAL13` will also be preserved as an optional feature where available.

# 31. Construct and save the final baseline-aligned ADAS cohort

In [ ]:
# Construct the participant-level baseline-aligned ADAS table

adas_selected_90d = adas_nearest.loc[
    adas_nearest["ABS_DAYS_FROM_BASELINE"] <= 90,
    [
        "RID",
        "PHASE",
        "VISCODE",
        "VISCODE2",
        "VISDATE",
        "TOTSCORE",
        "TOTAL13",
        "SOURCE",
        "LANGUAGE_CODE",
        "HAS_QC_ERROR",
        "DAYS_FROM_BASELINE",
        "ABS_DAYS_FROM_BASELINE",
        "flag_unresolved_qc_error",
        "flag_totscore_out_of_range",
        "flag_total13_out_of_range",
        "flag_provisional_exclusion",
    ],
].copy()

adas_baseline_aligned = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        adas_selected_90d,
        on="RID",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "PHASE": "ADAS_PHASE",
            "VISCODE": "ADAS_VISCODE",
            "VISCODE2": "ADAS_VISCODE2",
            "VISDATE": "ADAS_DATE",
            "SOURCE": "ADAS_SOURCE",
            "LANGUAGE_CODE": "ADAS_LANGUAGE_CODE",
            "HAS_QC_ERROR": "ADAS_HAS_QC_ERROR",
        }
    )
)

adas_baseline_aligned["ADAS_AVAILABLE"] = (
    adas_baseline_aligned["TOTSCORE"].notna()
)

print("Baseline-aligned ADAS cohort")
print("----------------------------")
print(f"Rows: {len(adas_baseline_aligned):,}")
print(f"Unique RID: {adas_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{adas_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "ADAS available within ±90 days:",
    f"{adas_baseline_aligned['ADAS_AVAILABLE'].sum():,}"
)
print(
    "ADAS unavailable:",
    f"{(~adas_baseline_aligned['ADAS_AVAILABLE']).sum():,}"
)
print(
    "Selected records missing TOTAL13:",
    f"{adas_baseline_aligned.loc[adas_baseline_aligned['ADAS_AVAILABLE'], 'TOTAL13'].isna().sum():,}"
)

ADAS_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "adas"
)

ADAS_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ADAS_FINAL_PATH = (
    ADAS_OUTPUT_DIR
    / "adas_baseline_aligned_cohort.csv"
)

adas_baseline_aligned.to_csv(
    ADAS_FINAL_PATH,
    index=False,
)

adas_reloaded = pd.read_csv(
    ADAS_FINAL_PATH,
    low_memory=False,
)

print("\nSaved ADAS file")
print("---------------")
print("Path:", ADAS_FINAL_PATH)
print("Exists:", ADAS_FINAL_PATH.exists())
print(f"Rows after reload: {len(adas_reloaded):,}")
print(f"Unique RID after reload: {adas_reloaded['RID'].nunique():,}")
print(f"Duplicate RID after reload: {adas_reloaded['RID'].duplicated().sum():,}")

# 32. Load the cleaned visit-level CSF core biomarker table

The cleaned CSF output is stored in:

`/content/drive/MyDrive/adni_mri/adni_non_imaging/interim/csf_core_biomarkers/csf_core_biomarkers_cleaned_visit_level.csv`

This visit-level table will be used to construct the final baseline-aligned CSF modality. Because CSF biomarkers are less frequently collected than cognitive assessments, their timing window will be evaluated separately before final selection.

In [ ]:
# Load the cleaned visit-level CSF core biomarker table

CSF_PATH = (
    INTERIM_DIR
    / "csf_core_biomarkers"
    / "csf_core_biomarkers_cleaned_visit_level.csv"
)

print("CSF path:")
print(CSF_PATH)
print("Exists:", CSF_PATH.exists())

if not CSF_PATH.exists():
    raise FileNotFoundError(
        f"The cleaned CSF file was not found: {CSF_PATH}"
    )

csf_cleaned = pd.read_csv(
    CSF_PATH,
    low_memory=False,
)

print(f"\nCSF rows: {len(csf_cleaned):,}")
print(f"CSF columns: {len(csf_cleaned.columns)}")

print("\nColumns:")
print(csf_cleaned.columns.tolist())

display(csf_cleaned.head())

# 33. Audit CSF coverage, biomarker availability, and quality-control flags

The cleaned CSF table contains visit-level measurements for:

- amyloid beta 40 (`ABETA40`);
- amyloid beta 42 (`ABETA42`);
- total tau (`TAU`);
- phosphorylated tau (`PTAU`);
- the derived amyloid ratio (`ABETA42_40_RATIO`).

Because `ABETA40` is structurally unavailable in the older `UPENNBIOMK9` batch, the ratio cannot be expected for every otherwise valid CSF sample. Coverage must therefore be assessed separately for each biomarker and for the ratio.

This step will merge CSF with the authoritative cohort, calculate timing relative to clinical baseline, and summarize quality-control flags.

In [ ]:
# Prepare CSF and audit cohort coverage and data quality

csf_cleaned["RID"] = pd.to_numeric(
    csf_cleaned["RID"],
    errors="coerce",
).astype("Int64")

csf_cleaned["EXAMDATE"] = pd.to_datetime(
    csf_cleaned["EXAMDATE"],
    errors="coerce",
)

csf_cleaned["RUNDATE"] = pd.to_datetime(
    csf_cleaned["RUNDATE"],
    errors="coerce",
)

for column in [
    "ABETA40",
    "ABETA42",
    "TAU",
    "PTAU",
    "ABETA42_40_RATIO",
]:
    csf_cleaned[column] = pd.to_numeric(
        csf_cleaned[column],
        errors="coerce",
    )

csf_cohort = csf_cleaned.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "FIRST_AD_DATE",
        ]
    ].rename(columns={"PTID": "COHORT_PTID"}),
    on="RID",
    how="inner",
    validate="many_to_one",
)

csf_cohort["BASELINE_DATE"] = pd.to_datetime(
    csf_cohort["BASELINE_DATE"],
    errors="coerce",
)

csf_cohort["FIRST_AD_DATE"] = pd.to_datetime(
    csf_cohort["FIRST_AD_DATE"],
    errors="coerce",
)

csf_cohort["DAYS_FROM_BASELINE"] = (
    csf_cohort["EXAMDATE"] - csf_cohort["BASELINE_DATE"]
).dt.days

csf_cohort["ABS_DAYS_FROM_BASELINE"] = (
    csf_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

csf_rids = set(
    csf_cleaned["RID"]
    .dropna()
    .astype(int)
)

matched_csf_rids = cohort_rids & csf_rids

print("CSF cohort coverage")
print("-------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in CSF: {len(matched_csf_rids):,}")
print(f"Participants absent from CSF: {len(cohort_rids - csf_rids):,}")

print("\nMatched CSF records")
print("-------------------")
print(f"Rows: {len(csf_cohort):,}")
print(f"Unique RID: {csf_cohort['RID'].nunique():,}")
print(f"Missing EXAMDATE: {csf_cohort['EXAMDATE'].isna().sum():,}")

print("\nBiomarker availability")
print("----------------------")
for column in [
    "ABETA40",
    "ABETA42",
    "TAU",
    "PTAU",
    "ABETA42_40_RATIO",
]:
    print(
        f"{column}: "
        f"{csf_cohort[column].notna().sum():,} records, "
        f"{csf_cohort.loc[csf_cohort[column].notna(), 'RID'].nunique():,} participants"
    )

print("\nQuality-control flags")
print("---------------------")
for column in [
    "ABETA42_BELOW_LIMIT",
    "ABETA42_ABOVE_LIMIT",
    "TAU_BELOW_LIMIT",
    "TAU_ABOVE_LIMIT",
    "PTAU_BELOW_LIMIT",
    "PTAU_ABOVE_LIMIT",
    "RECALCULATION_FAILED",
    "SAMPLE_HEMOLYZED",
    "ABETA40_STRUCTURALLY_MISSING",
    "NO_CORE_BIOMARKERS_AVAILABLE",
    "ABETA42_40_RATIO_IQR_OUTLIER",
]:
    print(
        f"{column}: "
        f"{csf_cohort[column].fillna(False).astype(bool).sum():,}"
    )

print("\nAvailability by batch")
print("---------------------")
display(
    csf_cohort.groupby("BATCH").agg(
        RECORDS=("RID", "size"),
        UNIQUE_RID=("RID", "nunique"),
        ABETA40_AVAILABLE=("ABETA40", "count"),
        ABETA42_AVAILABLE=("ABETA42", "count"),
        TAU_AVAILABLE=("TAU", "count"),
        PTAU_AVAILABLE=("PTAU", "count"),
        RATIO_AVAILABLE=("ABETA42_40_RATIO", "count"),
    )
)

print("\nParticipant coverage by cohort label")
print("------------------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_ANY_CSF=authoritative_clinical_cohort["RID"].isin(
            matched_csf_rids
        )
    )
    .groupby("COHORT_LABEL")["HAS_ANY_CSF"]
    .value_counts()
    .unstack(fill_value=0)
)

### 33.1. CSF coverage and biomarker-availability summary

CSF is available for 1,264 of the 2,199 cohort participants.

The core biomarkers have substantially different coverage:

- `ABETA42` is available for all 1,264 CSF participants;
- `TAU` is available for 1,259 participants;
- `PTAU` is available for 1,256 participants;
- `ABETA40` and the `ABETA42_40_RATIO` are available for only 436 participants because `ABETA40` is structurally absent from the older `UPENNBIOMK9` batch.

Values reported above or below assay limits will be preserved together with their flags rather than automatically discarded. These are assay-censoring indicators, not necessarily invalid measurements.

The next step is to select the nearest usable CSF sample to clinical baseline and evaluate coverage within a wider ±180-day window, reflecting the less frequent collection of lumbar-puncture biomarkers.

In [ ]:
# Select the nearest usable CSF assessment to clinical baseline

csf_eligible = csf_cohort.loc[
    csf_cohort["EXAMDATE"].notna()
    & ~csf_cohort["NO_CORE_BIOMARKERS_AVAILABLE"].fillna(False)
    & ~csf_cohort["RECALCULATION_FAILED"].fillna(False)
    & ~csf_cohort["SAMPLE_HEMOLYZED"].fillna(False)
    & (
        csf_cohort[
            ["ABETA42", "TAU", "PTAU", "ABETA40", "ABETA42_40_RATIO"]
        ]
        .notna()
        .any(axis=1)
    )
].copy()

# Prevent post-conversion measurements from being selected for pMCI
csf_eligible["IS_BEFORE_PMCICONVERSION"] = (
    ~csf_eligible["COHORT_LABEL"].eq("pMCI")
    | csf_eligible["FIRST_AD_DATE"].isna()
    | (csf_eligible["EXAMDATE"] < csf_eligible["FIRST_AD_DATE"])
)

csf_eligible = csf_eligible.loc[
    csf_eligible["IS_BEFORE_PMCICONVERSION"]
].copy()

# For equal absolute distance, prefer the pre-baseline sample
csf_eligible["POST_BASELINE_TIEBREAKER"] = (
    csf_eligible["DAYS_FROM_BASELINE"] > 0
).astype(int)

# Prefer records with more available biomarkers when dates are equally close
csf_nearest = (
    csf_eligible
    .sort_values(
        [
            "RID",
            "ABS_DAYS_FROM_BASELINE",
            "POST_BASELINE_TIEBREAKER",
            "AVAILABLE_CORE_BIOMARKER_COUNT",
            "EXAMDATE",
        ],
        ascending=[True, True, True, False, True],
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .copy()
)

csf_nearest["TIMING_DIRECTION"] = pd.cut(
    csf_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

selected_csf_rids = set(
    csf_nearest["RID"]
    .dropna()
    .astype(int)
)

print("Nearest usable CSF assessment")
print("-----------------------------")
print(f"Selected participants: {len(csf_nearest):,}")
print(f"Unique RID: {csf_nearest['RID'].nunique():,}")
print(
    "Cohort participants without a usable dated CSF sample:",
    f"{len(cohort_rids - selected_csf_rids):,}"
)

print("\nTiming direction")
print("----------------")
print(
    csf_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [30, 60, 90, 180, 365]:
    count = csf_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(authoritative_clinical_cohort)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}% of cohort)"
    )

print("\nNearest CSF timing summary")
print("--------------------------")
print(
    csf_nearest["ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nBiomarker availability within ±180 days")
print("----------------------------------------")

csf_within_180d = csf_nearest.loc[
    csf_nearest["ABS_DAYS_FROM_BASELINE"] <= 180
]

for column in [
    "ABETA42",
    "TAU",
    "PTAU",
    "ABETA40",
    "ABETA42_40_RATIO",
]:
    print(
        f"{column}: "
        f"{csf_within_180d[column].notna().sum():,} participants"
    )

print("\nCSF availability within ±180 days by cohort label")
print("-------------------------------------------------")
display(
    authoritative_clinical_cohort[
        ["RID", "COHORT_LABEL"]
    ]
    .assign(
        HAS_CSF_WITHIN_180D=lambda df: df["RID"].isin(
            set(csf_within_180d["RID"])
        )
    )
    .groupby("COHORT_LABEL")["HAS_CSF_WITHIN_180D"]
    .value_counts()
    .unstack(fill_value=0)
)

### 33.2. CSF baseline-alignment decision

CSF will be treated as available at baseline when the nearest usable sample falls within ±180 days of the authoritative DXSUM baseline.

This gives baseline-aligned CSF coverage for 1,225 of the 2,199 participants. The remaining 974 participants will remain in the cohort with CSF marked as unavailable.

Because `ABETA40` is structurally absent from the older `UPENNBIOMK9` batch, CSF availability and amyloid-ratio availability will be represented separately:

- `CSF_AVAILABLE` indicates that at least one core CSF biomarker is available within ±180 days;
- `CSF_RATIO_AVAILABLE` indicates that `ABETA42_40_RATIO` is available;
- assay-limit flags will be retained rather than treating censored values as ordinary missing data.

# 34. Construct and save the final baseline-aligned CSF cohort

In [ ]:
# Construct the participant-level baseline-aligned CSF table

csf_selected_180d = csf_nearest.loc[
    csf_nearest["ABS_DAYS_FROM_BASELINE"] <= 180,
    [
        "RID",
        "PHASE",
        "VISCODE2",
        "EXAMDATE",
        "BATCH",
        "RUNDATE",
        "ABETA40",
        "ABETA42",
        "TAU",
        "PTAU",
        "ABETA42_40_RATIO",
        "AVAILABLE_CORE_BIOMARKER_COUNT",
        "DAYS_FROM_BASELINE",
        "ABS_DAYS_FROM_BASELINE",
        "ABETA42_BELOW_LIMIT",
        "ABETA42_ABOVE_LIMIT",
        "TAU_BELOW_LIMIT",
        "TAU_ABOVE_LIMIT",
        "PTAU_BELOW_LIMIT",
        "PTAU_ABOVE_LIMIT",
        "ABETA40_STRUCTURALLY_MISSING",
        "ABETA42_40_RATIO_AVAILABLE",
        "ABETA42_40_RATIO_IQR_OUTLIER",
        "RECALCULATION_FAILED",
        "SAMPLE_HEMOLYZED",
    ],
].copy()

csf_baseline_aligned = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        csf_selected_180d,
        on="RID",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "PHASE": "CSF_PHASE",
            "VISCODE2": "CSF_VISCODE2",
            "EXAMDATE": "CSF_DATE",
            "BATCH": "CSF_BATCH",
            "RUNDATE": "CSF_RUNDATE",
        }
    )
)

csf_baseline_aligned["CSF_AVAILABLE"] = (
    csf_baseline_aligned[
        ["ABETA42", "TAU", "PTAU", "ABETA40", "ABETA42_40_RATIO"]
    ]
    .notna()
    .any(axis=1)
)

csf_baseline_aligned["CSF_RATIO_AVAILABLE"] = (
    csf_baseline_aligned["ABETA42_40_RATIO"].notna()
)

print("Baseline-aligned CSF cohort")
print("---------------------------")
print(f"Rows: {len(csf_baseline_aligned):,}")
print(f"Unique RID: {csf_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{csf_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "CSF available within ±180 days:",
    f"{csf_baseline_aligned['CSF_AVAILABLE'].sum():,}"
)
print(
    "CSF unavailable:",
    f"{(~csf_baseline_aligned['CSF_AVAILABLE']).sum():,}"
)
print(
    "CSF ratio available:",
    f"{csf_baseline_aligned['CSF_RATIO_AVAILABLE'].sum():,}"
)

CSF_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "csf_core_biomarkers"
)

CSF_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CSF_FINAL_PATH = (
    CSF_OUTPUT_DIR
    / "csf_core_biomarkers_baseline_aligned_cohort.csv"
)

csf_baseline_aligned.to_csv(
    CSF_FINAL_PATH,
    index=False,
)

csf_reloaded = pd.read_csv(
    CSF_FINAL_PATH,
    low_memory=False,
)

print("\nSaved CSF file")
print("--------------")
print("Path:", CSF_FINAL_PATH)
print("Exists:", CSF_FINAL_PATH.exists())
print(f"Rows after reload: {len(csf_reloaded):,}")
print(f"Unique RID after reload: {csf_reloaded['RID'].nunique():,}")
print(f"Duplicate RID after reload: {csf_reloaded['RID'].duplicated().sum():,}")

# 35. Load the cleaned longitudinal plasma biomarker table

The cleaned plasma output is stored in:

`/content/drive/MyDrive/adni_mri/adni_non_imaging/processed/plasma/plasma_biomarkers_cleaned_longitudinal.csv`

This longitudinal table will be used to construct the final baseline-aligned plasma modality. Because plasma biomarkers are time-varying but generally more frequently collected than CSF, their timing distribution will be inspected before fixing the final eligibility window.

In [ ]:
# Load the cleaned longitudinal plasma biomarker table

PLASMA_PATH = (
    PROCESSED_DIR
    / "plasma"
    / "plasma_biomarkers_cleaned_longitudinal.csv"
)

print("Plasma path:")
print(PLASMA_PATH)
print("Exists:", PLASMA_PATH.exists())

if not PLASMA_PATH.exists():
    raise FileNotFoundError(
        f"The cleaned plasma file was not found: {PLASMA_PATH}"
    )

plasma_cleaned = pd.read_csv(
    PLASMA_PATH,
    low_memory=False,
)

print(f"\nPlasma rows: {len(plasma_cleaned):,}")
print(f"Plasma columns: {len(plasma_cleaned.columns)}")

print("\nColumns:")
print(plasma_cleaned.columns.tolist())

display(plasma_cleaned.head())

# 36. Audit plasma coverage, biomarker availability, and quality-control flags

The cleaned plasma table contains several biomarker groups:

- Fujirebio amyloid and phosphorylated tau:
  - `pT217_F`
  - `AB42_F`
  - `AB40_F`
  - `AB42_AB40_F`
  - `pT217_AB42_F`
- Quanterix neurodegeneration and astrocytic markers:
  - `NfL_Q`
  - `GFAP_Q`
- Fujirebio neurodegeneration and astrocytic markers:
  - `NfL_F`
  - `GFAP_F`

Coverage and quality-control status will be assessed separately for each assay group. Records flagged as extreme rounded amyloid values will not be used for amyloid-related baseline features, while validated Batch 3 drift records will remain eligible with their QC indicator preserved.

In [ ]:
# Prepare plasma and audit cohort coverage and data quality

plasma_cleaned["RID"] = pd.to_numeric(
    plasma_cleaned["RID"],
    errors="coerce",
).astype("Int64")

plasma_cleaned["EXAMDATE"] = pd.to_datetime(
    plasma_cleaned["EXAMDATE"],
    errors="coerce",
)

plasma_biomarker_columns = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
]

for column in plasma_biomarker_columns:
    plasma_cleaned[column] = pd.to_numeric(
        plasma_cleaned[column],
        errors="coerce",
    )

plasma_cohort = plasma_cleaned.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "BASELINE_PHASE",
            "BASELINE_DATE",
            "COHORT_LABEL",
            "FIRST_AD_DATE",
        ]
    ].rename(columns={"PTID": "COHORT_PTID"}),
    on="RID",
    how="inner",
    validate="many_to_one",
)

plasma_cohort["BASELINE_DATE"] = pd.to_datetime(
    plasma_cohort["BASELINE_DATE"],
    errors="coerce",
)

plasma_cohort["FIRST_AD_DATE"] = pd.to_datetime(
    plasma_cohort["FIRST_AD_DATE"],
    errors="coerce",
)

plasma_cohort["DAYS_FROM_BASELINE"] = (
    plasma_cohort["EXAMDATE"] - plasma_cohort["BASELINE_DATE"]
).dt.days

plasma_cohort["ABS_DAYS_FROM_BASELINE"] = (
    plasma_cohort["DAYS_FROM_BASELINE"].abs()
)

cohort_rids = set(
    authoritative_clinical_cohort["RID"]
    .dropna()
    .astype(int)
)

plasma_rids = set(
    plasma_cleaned["RID"]
    .dropna()
    .astype(int)
)

matched_plasma_rids = cohort_rids & plasma_rids

print("Plasma cohort coverage")
print("----------------------")
print(f"Authoritative cohort participants: {len(cohort_rids):,}")
print(f"Participants present in plasma: {len(matched_plasma_rids):,}")
print(f"Participants absent from plasma: {len(cohort_rids - plasma_rids):,}")

print("\nMatched plasma records")
print("----------------------")
print(f"Rows: {len(plasma_cohort):,}")
print(f"Unique RID: {plasma_cohort['RID'].nunique():,}")
print(f"Missing EXAMDATE: {plasma_cohort['EXAMDATE'].isna().sum():,}")

print("\nBiomarker availability")
print("----------------------")
for column in plasma_biomarker_columns:
    print(
        f"{column}: "
        f"{plasma_cohort[column].notna().sum():,} records, "
        f"{plasma_cohort.loc[plasma_cohort[column].notna(), 'RID'].nunique():,} participants"
    )

print("\nAssay-group availability")
print("------------------------")
for column in [
    "HAS_FUJIREBIO_AMYLOID_PTAU",
    "HAS_QUANTERIX_NFL_GFAP",
    "HAS_FUJIREBIO_NFL_GFAP",
]:
    print(
        f"{column}: "
        f"{plasma_cohort[column].fillna(False).astype(bool).sum():,} records"
    )

print("\nQuality-control flags")
print("---------------------")
for column in [
    "BATCH3_QC_DRIFT_VALIDATED",
    "EXTREME_ROUND_AMYLOID_EXCLUDED",
]:
    print(
        f"{column}: "
        f"{plasma_cohort[column].fillna(False).astype(bool).sum():,}"
    )

print("\nParticipant coverage by cohort label")
print("------------------------------------")
display(
    authoritative_clinical_cohort.assign(
        HAS_ANY_PLASMA=authoritative_clinical_cohort["RID"].isin(
            matched_plasma_rids
        )
    )
    .groupby("COHORT_LABEL")["HAS_ANY_PLASMA"]
    .value_counts()
    .unstack(fill_value=0)
)

### 36.1. Plasma coverage and quality-control summary

Plasma is available for 1,217 of the 2,199 cohort participants.

Among participants with plasma data, Fujirebio amyloid and p-tau measurements are nearly complete. NfL and GFAP coverage varies by assay platform. The 276 Batch 3 records with validated QC drift will remain eligible with their indicator preserved, while the single record flagged as an extreme rounded amyloid value will not contribute amyloid-related features.

The next step is to select the nearest usable plasma assessment to the authoritative clinical baseline and inspect coverage within a candidate ±180-day window.

In [ ]:
# Select the nearest usable plasma assessment to clinical baseline

plasma_eligible = plasma_cohort.loc[
    plasma_cohort["EXAMDATE"].notna()
    & (
        plasma_cohort[plasma_biomarker_columns]
        .notna()
        .any(axis=1)
    )
].copy()

# Prevent post-conversion measurements from being selected for pMCI
plasma_eligible["IS_BEFORE_PMCICONVERSION"] = (
    ~plasma_eligible["COHORT_LABEL"].eq("pMCI")
    | plasma_eligible["FIRST_AD_DATE"].isna()
    | (plasma_eligible["EXAMDATE"] < plasma_eligible["FIRST_AD_DATE"])
)

plasma_eligible = plasma_eligible.loc[
    plasma_eligible["IS_BEFORE_PMCICONVERSION"]
].copy()

# For equal absolute distance, prefer the pre-baseline sample
plasma_eligible["POST_BASELINE_TIEBREAKER"] = (
    plasma_eligible["DAYS_FROM_BASELINE"] > 0
).astype(int)

# Count available biomarkers to break any remaining timing ties
plasma_eligible["AVAILABLE_PLASMA_BIOMARKER_COUNT"] = (
    plasma_eligible[plasma_biomarker_columns]
    .notna()
    .sum(axis=1)
)

plasma_nearest = (
    plasma_eligible
    .sort_values(
        [
            "RID",
            "ABS_DAYS_FROM_BASELINE",
            "POST_BASELINE_TIEBREAKER",
            "AVAILABLE_PLASMA_BIOMARKER_COUNT",
            "EXAMDATE",
        ],
        ascending=[True, True, True, False, True],
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .copy()
)

plasma_nearest["TIMING_DIRECTION"] = pd.cut(
    plasma_nearest["DAYS_FROM_BASELINE"],
    bins=[-float("inf"), -1, 0, float("inf")],
    labels=["Before baseline", "Same day", "After baseline"],
)

selected_plasma_rids = set(
    plasma_nearest["RID"]
    .dropna()
    .astype(int)
)

print("Nearest usable plasma assessment")
print("--------------------------------")
print(f"Selected participants: {len(plasma_nearest):,}")
print(f"Unique RID: {plasma_nearest['RID'].nunique():,}")
print(
    "Cohort participants without a usable dated plasma sample:",
    f"{len(cohort_rids - selected_plasma_rids):,}"
)

print("\nTiming direction")
print("----------------")
print(
    plasma_nearest["TIMING_DIRECTION"]
    .value_counts(dropna=False)
)

print("\nDistance from clinical baseline")
print("-------------------------------")
for window in [30, 60, 90, 180, 365]:
    count = plasma_nearest["ABS_DAYS_FROM_BASELINE"].le(window).sum()
    percentage = 100 * count / len(authoritative_clinical_cohort)

    print(
        f"Within ±{window} days: "
        f"{count:,} ({percentage:.2f}% of cohort)"
    )

print("\nNearest plasma timing summary")
print("-----------------------------")
print(
    plasma_nearest["ABS_DAYS_FROM_BASELINE"]
    .describe()
)

print("\nBiomarker availability within ±180 days")
print("----------------------------------------")

plasma_within_180d = plasma_nearest.loc[
    plasma_nearest["ABS_DAYS_FROM_BASELINE"] <= 180
]

for column in plasma_biomarker_columns:
    print(
        f"{column}: "
        f"{plasma_within_180d[column].notna().sum():,} participants"
    )

print("\nPlasma availability within ±180 days by cohort label")
print("----------------------------------------------------")
display(
    authoritative_clinical_cohort[
        ["RID", "COHORT_LABEL"]
    ]
    .assign(
        HAS_PLASMA_WITHIN_180D=lambda df: df["RID"].isin(
            set(plasma_within_180d["RID"])
        )
    )
    .groupby("COHORT_LABEL")["HAS_PLASMA_WITHIN_180D"]
    .value_counts()
    .unstack(fill_value=0)
)

# 37. Construct and save the final baseline-aligned plasma cohort

In [ ]:
# Construct the participant-level baseline-aligned plasma table

plasma_selected_180d = plasma_nearest.loc[
    plasma_nearest["ABS_DAYS_FROM_BASELINE"] <= 180,
    [
        "RID",
        "PHASE",
        "VISCODE",
        "VISCODE2",
        "EXAMDATE",
        "Primary",
        "Additive",
        "pT217_F",
        "AB42_F",
        "AB40_F",
        "AB42_AB40_F",
        "pT217_AB42_F",
        "NfL_Q",
        "GFAP_Q",
        "NfL_F",
        "GFAP_F",
        "HAS_FUJIREBIO_AMYLOID_PTAU",
        "HAS_QUANTERIX_NFL_GFAP",
        "HAS_FUJIREBIO_NFL_GFAP",
        "BATCH3_QC_DRIFT_VALIDATED",
        "EXTREME_ROUND_AMYLOID_EXCLUDED",
        "AMYLOID_QC_REASON",
        "Comment",
        "AVAILABLE_PLASMA_BIOMARKER_COUNT",
        "DAYS_FROM_BASELINE",
        "ABS_DAYS_FROM_BASELINE",
    ],
].copy()

# Remove only the affected amyloid/p-tau values from the flagged record
amyloid_columns = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
]

extreme_amyloid_mask = (
    plasma_selected_180d["EXTREME_ROUND_AMYLOID_EXCLUDED"]
    .fillna(False)
    .astype(bool)
)

plasma_selected_180d.loc[
    extreme_amyloid_mask,
    amyloid_columns,
] = pd.NA

plasma_baseline_aligned = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        plasma_selected_180d,
        on="RID",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "PHASE": "PLASMA_PHASE",
            "VISCODE": "PLASMA_VISCODE",
            "VISCODE2": "PLASMA_VISCODE2",
            "EXAMDATE": "PLASMA_DATE",
            "Primary": "PLASMA_PRIMARY",
            "Additive": "PLASMA_ADDITIVE",
            "Comment": "PLASMA_COMMENT",
        }
    )
)

plasma_baseline_aligned["PLASMA_AVAILABLE"] = (
    plasma_baseline_aligned[plasma_biomarker_columns]
    .notna()
    .any(axis=1)
)

plasma_baseline_aligned["PLASMA_AMYLOID_PTAU_AVAILABLE"] = (
    plasma_baseline_aligned[amyloid_columns]
    .notna()
    .any(axis=1)
)

plasma_baseline_aligned["PLASMA_QUANTERIX_NFL_GFAP_AVAILABLE"] = (
    plasma_baseline_aligned[["NfL_Q", "GFAP_Q"]]
    .notna()
    .any(axis=1)
)

plasma_baseline_aligned["PLASMA_FUJIREBIO_NFL_GFAP_AVAILABLE"] = (
    plasma_baseline_aligned[["NfL_F", "GFAP_F"]]
    .notna()
    .any(axis=1)
)

print("Baseline-aligned plasma cohort")
print("------------------------------")
print(f"Rows: {len(plasma_baseline_aligned):,}")
print(f"Unique RID: {plasma_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{plasma_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "Any plasma available within ±180 days:",
    f"{plasma_baseline_aligned['PLASMA_AVAILABLE'].sum():,}"
)
print(
    "Amyloid/p-tau available:",
    f"{plasma_baseline_aligned['PLASMA_AMYLOID_PTAU_AVAILABLE'].sum():,}"
)
print(
    "Quanterix NfL/GFAP available:",
    f"{plasma_baseline_aligned['PLASMA_QUANTERIX_NFL_GFAP_AVAILABLE'].sum():,}"
)
print(
    "Fujirebio NfL/GFAP available:",
    f"{plasma_baseline_aligned['PLASMA_FUJIREBIO_NFL_GFAP_AVAILABLE'].sum():,}"
)

PLASMA_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "plasma"
)

PLASMA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PLASMA_FINAL_PATH = (
    PLASMA_OUTPUT_DIR
    / "plasma_baseline_aligned_cohort.csv"
)

plasma_baseline_aligned.to_csv(
    PLASMA_FINAL_PATH,
    index=False,
)

plasma_reloaded = pd.read_csv(
    PLASMA_FINAL_PATH,
    low_memory=False,
)

print("\nSaved plasma file")
print("-----------------")
print("Path:", PLASMA_FINAL_PATH)
print("Exists:", PLASMA_FINAL_PATH.exists())
print(f"Rows after reload: {len(plasma_reloaded):,}")
print(f"Unique RID after reload: {plasma_reloaded['RID'].nunique():,}")
print(f"Duplicate RID after reload: {plasma_reloaded['RID'].duplicated().sum():,}")

# 38. Load the cleaned participant-level APOE table

The cleaned APOE genotype table is stored at:

`/content/drive/MyDrive/adni_mri/adni_non_imaging/processed/apoe/apoe_genotype_cleaned_participant_level.csv`

APOE is genetically stable, so no temporal baseline window is required. The cleaned participant-level table will be merged directly with the authoritative clinical cohort using `RID`.

In [ ]:
# Load the cleaned participant-level APOE table

APOE_PATH = (
    PROCESSED_DIR
    / "apoe"
    / "apoe_genotype_cleaned_participant_level.csv"
)

print("APOE path:")
print(APOE_PATH)
print("Exists:", APOE_PATH.exists())

if not APOE_PATH.exists():
    raise FileNotFoundError(
        f"The cleaned APOE file was not found: {APOE_PATH}"
    )

apoe_cleaned = pd.read_csv(
    APOE_PATH,
    low_memory=False,
)

print(f"\nAPOE rows: {len(apoe_cleaned):,}")
print(f"APOE columns: {len(apoe_cleaned.columns)}")
print(f"Unique RID: {apoe_cleaned['RID'].nunique():,}")
print(f"Duplicate RID: {apoe_cleaned['RID'].duplicated().sum():,}")

print("\nColumns:")
print(apoe_cleaned.columns.tolist())

display(apoe_cleaned.head())

# 39. Merge APOE with the authoritative clinical cohort

The cleaned APOE table contains one valid genotype record per participant and requires no temporal filtering.

This step will merge APOE directly with the authoritative 2,199-participant cohort using `RID`, confirm coverage by diagnostic group, and retain participants without APOE as missing rather than excluding them.

In [ ]:
# Merge APOE into the authoritative clinical cohort

apoe_cleaned["RID"] = pd.to_numeric(
    apoe_cleaned["RID"],
    errors="coerce",
).astype("Int64")

apoe_cohort = (
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ]
    .merge(
        apoe_cleaned,
        on="RID",
        how="left",
        validate="one_to_one",
        suffixes=("", "_APOE"),
    )
)

apoe_cohort["APOE_AVAILABLE"] = (
    apoe_cohort["APOE_GENOTYPE"].notna()
)

print("APOE cohort coverage")
print("--------------------")
print(f"Rows: {len(apoe_cohort):,}")
print(f"Unique RID: {apoe_cohort['RID'].nunique():,}")
print(f"Duplicate RID: {apoe_cohort['RID'].duplicated().sum():,}")
print(f"APOE available: {apoe_cohort['APOE_AVAILABLE'].sum():,}")
print(f"APOE unavailable: {(~apoe_cohort['APOE_AVAILABLE']).sum():,}")

print("\nCoverage by cohort label")
print("------------------------")
display(
    pd.crosstab(
        apoe_cohort["COHORT_LABEL"],
        apoe_cohort["APOE_AVAILABLE"],
        margins=True,
    )
)

print("\nGenotype distribution within the authoritative cohort")
print("-----------------------------------------------------")
display(
    apoe_cohort.loc[
        apoe_cohort["APOE_AVAILABLE"],
        "APOE_GENOTYPE",
    ]
    .value_counts()
    .rename_axis("APOE_GENOTYPE")
    .reset_index(name="PARTICIPANTS")
)

print("\nAPOE ε4 allele-count distribution")
print("---------------------------------")
display(
    apoe_cohort.loc[
        apoe_cohort["APOE_AVAILABLE"],
        "APOE4_ALLELE_COUNT",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("APOE4_ALLELE_COUNT")
    .reset_index(name="PARTICIPANTS")
)

### 39.1. APOE cohort coverage summary

APOE genotype is available for 2,126 of the 2,199 participants in the authoritative cohort.

Coverage is especially high for the MCI groups:

- 248 of 249 pMCI participants;
- all 295 sMCI participants;
- 452 of 459 AD participants;
- 1,131 of 1,196 CN participants.

The genotype and ε4 allele-count distributions are internally consistent. Before saving the final APOE manifest, the cohort and APOE participant identifiers will be checked for mismatches, and all retained genotype records will be revalidated.

In [ ]:
# Validate APOE records after merging with the authoritative cohort

apoe_available = apoe_cohort["APOE_AVAILABLE"]

ptid_mismatch = (
    apoe_available
    & apoe_cohort["PTID_APOE"].notna()
    & (
        apoe_cohort["PTID"].astype("string")
        != apoe_cohort["PTID_APOE"].astype("string")
    )
)

invalid_genotype = (
    apoe_available
    & ~apoe_cohort["APOE_GENOTYPE_VALID"].fillna(False).astype(bool)
)

unresolved_conflict = (
    apoe_available
    & apoe_cohort["APOE_UNRESOLVED_CONFLICT"]
    .fillna(False)
    .astype(bool)
)

unusable_sample = (
    apoe_available
    & ~apoe_cohort["APOE_SAMPLE_USABLE"]
    .fillna(False)
    .astype(bool)
)

allele_count_mismatch = (
    apoe_available
    & (
        (
            apoe_cohort["APOE_ALLELE_1"].eq(4).astype(int)
            + apoe_cohort["APOE_ALLELE_2"].eq(4).astype(int)
        )
        != apoe_cohort["APOE4_ALLELE_COUNT"]
    )
)

carrier_mismatch = (
    apoe_available
    & (
        apoe_cohort["APOE4_CARRIER"].astype("boolean")
        != apoe_cohort["APOE4_ALLELE_COUNT"].gt(0).astype("boolean")
    )
)

print("APOE merged-cohort validation")
print("-----------------------------")
print(f"PTID mismatch: {ptid_mismatch.sum():,}")
print(f"Invalid genotype: {invalid_genotype.sum():,}")
print(f"Unresolved conflict: {unresolved_conflict.sum():,}")
print(f"Unusable sample: {unusable_sample.sum():,}")
print(f"ε4 allele-count mismatch: {allele_count_mismatch.sum():,}")
print(f"ε4 carrier-status mismatch: {carrier_mismatch.sum():,}")

validation_issue_mask = (
    ptid_mismatch
    | invalid_genotype
    | unresolved_conflict
    | unusable_sample
    | allele_count_mismatch
    | carrier_mismatch
)

print(
    "Participants with any APOE validation issue:",
    f"{validation_issue_mask.sum():,}"
)

if validation_issue_mask.any():
    display(
        apoe_cohort.loc[
            validation_issue_mask,
            [
                "RID",
                "PTID",
                "PTID_APOE",
                "COHORT_LABEL",
                "APOE_ALLELE_1",
                "APOE_ALLELE_2",
                "APOE_GENOTYPE",
                "APOE4_ALLELE_COUNT",
                "APOE4_CARRIER",
                "APOE_GENOTYPE_VALID",
                "APOE_SAMPLE_USABLE",
                "APOE_UNRESOLVED_CONFLICT",
            ],
        ]
    )

### 39.2. Correct interpretation of `APOE_SAMPLE_USABLE`

The 1,487 apparent “unusable” samples are not truly unusable.

`APOE_SAMPLE_USABLE` is missing (`NaN`) for participants without detailed laboratory QC information. The previous validation incorrectly converted these missing values to `False`, which made them look unusable.

Only an explicit `False` should count as an unusable sample. Missing QC should remain unknown, especially because the genotype itself is valid and conflict-free.

In [ ]:
# Revalidate APOE sample usability without treating missing QC as failure

apoe_sample_usable = apoe_cohort["APOE_SAMPLE_USABLE"].astype("boolean")

explicitly_unusable_sample = (
    apoe_available
    & apoe_sample_usable.eq(False).fillna(False)
)

sample_usability_unknown = (
    apoe_available
    & apoe_sample_usable.isna()
)

invalid_genotype = (
    apoe_available
    & apoe_cohort["APOE_GENOTYPE_VALID"]
    .astype("boolean")
    .eq(False)
    .fillna(False)
)

unresolved_conflict = (
    apoe_available
    & apoe_cohort["APOE_UNRESOLVED_CONFLICT"]
    .astype("boolean")
    .eq(True)
    .fillna(False)
)

validation_issue_mask = (
    ptid_mismatch
    | invalid_genotype
    | unresolved_conflict
    | explicitly_unusable_sample
    | allele_count_mismatch
    | carrier_mismatch
)

print("Corrected APOE merged-cohort validation")
print("----------------------------------------")
print(f"PTID mismatch: {ptid_mismatch.sum():,}")
print(f"Invalid genotype: {invalid_genotype.sum():,}")
print(f"Unresolved conflict: {unresolved_conflict.sum():,}")
print(f"Explicitly unusable sample: {explicitly_unusable_sample.sum():,}")
print(f"Sample usability unknown: {sample_usability_unknown.sum():,}")
print(f"ε4 allele-count mismatch: {allele_count_mismatch.sum():,}")
print(f"ε4 carrier-status mismatch: {carrier_mismatch.sum():,}")
print(
    "Participants with any genuine APOE validation issue:",
    f"{validation_issue_mask.sum():,}"
)

### 39.3. APOE validation result

The APOE merge is valid.

Among the 2,126 cohort participants with APOE data:

- no PTID mismatches were found;
- all genotypes are valid;
- no unresolved conflicts remain;
- no samples are explicitly marked unusable;
- ε4 allele counts and carrier-status flags are internally consistent.

The 1,487 missing `APOE_SAMPLE_USABLE` values indicate unavailable laboratory QC metadata, not failed genotype samples. APOE can therefore be retained for all 2,126 participants with a reported genotype.

# 40. Finalise and save the APOE cohort

In [ ]:
# Construct the final participant-level APOE manifest

apoe_final_columns = [
    "RID",
    "PTID",
    "COHORT_LABEL",
    "BASELINE_PHASE",
    "BASELINE_DATE",
    "PHASE",
    "VISCODE",
    "APTESTDT",
    "APOE_ALLELE_1",
    "APOE_ALLELE_2",
    "APOE_GENOTYPE",
    "APOE4_ALLELE_COUNT",
    "APOE4_CARRIER",
    "APOE_GENOTYPE_VALID",
    "APOE_LAB_QC_AVAILABLE",
    "APOE_SAMPLE_USABLE",
    "APOE_RESAMPLE_REQUESTED",
    "APOE_RECEIVED_WITHIN_24H",
    "APOE_SHIPPED_AMBIENT",
    "APOE_UNRESOLVED_CONFLICT",
    "APOE_AVAILABLE",
]

apoe_baseline_aligned = (
    apoe_cohort[apoe_final_columns]
    .rename(
        columns={
            "PHASE": "APOE_PHASE",
            "VISCODE": "APOE_VISCODE",
            "APTESTDT": "APOE_TEST_DATE",
        }
    )
    .sort_values("RID")
    .reset_index(drop=True)
)

print("Final APOE cohort")
print("-----------------")
print(f"Rows: {len(apoe_baseline_aligned):,}")
print(f"Unique RID: {apoe_baseline_aligned['RID'].nunique():,}")
print(
    "Duplicate RID:",
    f"{apoe_baseline_aligned['RID'].duplicated().sum():,}"
)
print(
    "APOE available:",
    f"{apoe_baseline_aligned['APOE_AVAILABLE'].sum():,}"
)
print(
    "APOE unavailable:",
    f"{(~apoe_baseline_aligned['APOE_AVAILABLE']).sum():,}"
)

APOE_OUTPUT_DIR = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "apoe"
)

APOE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

APOE_FINAL_PATH = (
    APOE_OUTPUT_DIR
    / "apoe_baseline_aligned_cohort.csv"
)

apoe_baseline_aligned.to_csv(
    APOE_FINAL_PATH,
    index=False,
)

apoe_reloaded = pd.read_csv(
    APOE_FINAL_PATH,
    low_memory=False,
)

print("\nSaved APOE file")
print("---------------")
print("Path:", APOE_FINAL_PATH)
print("Exists:", APOE_FINAL_PATH.exists())
print(f"Rows after reload: {len(apoe_reloaded):,}")
print(f"Unique RID after reload: {apoe_reloaded['RID'].nunique():,}")
print(f"Duplicate RID after reload: {apoe_reloaded['RID'].duplicated().sum():,}")

# 41. Load and validate the existing MRI manifest

The previously completed MRI manifest is stored at:

`/content/drive/MyDrive/adni_mri/manifest/clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_manifest_1063.csv`

This step will verify that the manifest:

- contains exactly one row per MRI participant;
- has no missing or duplicated `RID`;
- preserves the expected 1,063-subject MRI cohort;
- has valid file paths and required preprocessing metadata;
- aligns consistently with the authoritative clinical cohort where participants overlap.

In [ ]:
# Load and validate the completed MRI manifest

MRI_MANIFEST_PATH = Path(
    "/content/drive/MyDrive/adni_mri/manifest/"
    "clean_90d_mni_n4_syn_rectcrop_177x213x183_t1norm_p01p99_manifest_1063.csv"
)

print("MRI manifest path:")
print(MRI_MANIFEST_PATH)
print("Exists:", MRI_MANIFEST_PATH.exists())

if not MRI_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"The MRI manifest was not found: {MRI_MANIFEST_PATH}"
    )

mri_manifest = pd.read_csv(
    MRI_MANIFEST_PATH,
    low_memory=False,
)

print(f"\nMRI manifest rows: {len(mri_manifest):,}")
print(f"MRI manifest columns: {len(mri_manifest.columns)}")

print("\nColumns:")
print(mri_manifest.columns.tolist())

display(mri_manifest.head())

# 42. Validate MRI manifest structure and alignment with the authoritative cohort

The MRI manifest contains the expected 1,063 rows and the complete preprocessing metadata.

This step will verify:

- one row per `RID`;
- no missing participant identifiers;
- MRI selection within ±90 days of the manifest baseline;
- required normalized T1 outputs and preprocessing QC;
- overlap with the reconstructed authoritative clinical cohort;
- agreement in PTID, cohort label, baseline date, and baseline phase.

The existing MRI manifest will not be modified during this audit.

In [ ]:
# Validate MRI manifest integrity and authoritative-cohort alignment

mri_manifest["RID"] = pd.to_numeric(
    mri_manifest["RID"],
    errors="coerce",
).astype("Int64")

mri_manifest["baseline_date"] = pd.to_datetime(
    mri_manifest["baseline_date"],
    errors="coerce",
)

mri_manifest["study_date"] = pd.to_datetime(
    mri_manifest["study_date"],
    errors="coerce",
)

mri_manifest["days_from_baseline_mri"] = pd.to_numeric(
    mri_manifest["days_from_baseline_mri"],
    errors="coerce",
)

mri_manifest["abs_days_from_baseline_mri"] = pd.to_numeric(
    mri_manifest["abs_days_from_baseline_mri"],
    errors="coerce",
)

# Basic participant-level integrity
print("MRI manifest participant integrity")
print("----------------------------------")
print(f"Rows: {len(mri_manifest):,}")
print(f"Unique RID: {mri_manifest['RID'].nunique(dropna=True):,}")
print(f"Missing RID: {mri_manifest['RID'].isna().sum():,}")
print(f"Duplicate RID: {mri_manifest['RID'].duplicated().sum():,}")
print(f"Missing PTID: {mri_manifest['PTID'].isna().sum():,}")
print(f"Missing baseline date: {mri_manifest['baseline_date'].isna().sum():,}")
print(f"Missing MRI study date: {mri_manifest['study_date'].isna().sum():,}")

# Recalculate MRI timing as an independent check
mri_manifest["RECALCULATED_DAYS_FROM_BASELINE"] = (
    mri_manifest["study_date"] - mri_manifest["baseline_date"]
).dt.days

timing_difference = (
    mri_manifest["RECALCULATED_DAYS_FROM_BASELINE"]
    != mri_manifest["days_from_baseline_mri"]
)

print("\nMRI timing validation")
print("---------------------")
print(
    "Stored versus recalculated day mismatch:",
    f"{timing_difference.fillna(True).sum():,}"
)
print(
    "Outside ±90 days:",
    f"{mri_manifest['abs_days_from_baseline_mri'].gt(90).sum():,}"
)
print(
    "Maximum absolute distance:",
    mri_manifest["abs_days_from_baseline_mri"].max(),
)

# Required preprocessing and file-status checks
print("\nMRI preprocessing validation")
print("----------------------------")

boolean_checks = [
    "image_found",
    "ras_nifti_path_exists",
    "input_ras_exists",
    "mni_nii_exists",
    "mni_npy_exists",
    "n4_nii_exists",
    "logjacobian_nii_exists",
    "logjacobian_npy_exists",
    "syn_qc_pass",
    "cropped_t1_nii_exists",
    "cropped_logjacobian_nii_exists",
    "cropped_t1_npy_exists",
    "cropped_logjacobian_npy_exists",
    "normalized_t1_nii_exists",
    "normalized_t1_npy_exists",
    "normalized_t1_npy_valid",
]

for column in boolean_checks:
    values = mri_manifest[column].astype("boolean")
    failed = values.eq(False).fillna(False).sum()
    missing = values.isna().sum()

    print(
        f"{column}: "
        f"failed={failed:,}, missing={missing:,}"
    )

print("\nCategorical QC/status distributions")
print("-----------------------------------")
for column in [
    "final_ras_qc_status",
    "crop_status",
    "t1_normalization_status",
    "output_shape",
    "mni_output_shape",
    "crop_shape",
    "normalized_t1_shape",
    "mni_template_used",
    "mni_registration_type",
    "n4_bias_correction",
    "t1_norm_method",
]:
    print(f"\n{column}:")
    print(mri_manifest[column].value_counts(dropna=False))

# Merge against reconstructed authoritative cohort
mri_authoritative_check = mri_manifest.merge(
    authoritative_clinical_cohort[
        [
            "RID",
            "PTID",
            "COHORT_LABEL",
            "BASELINE_PHASE",
            "BASELINE_DATE",
        ]
    ].rename(
        columns={
            "PTID": "AUTHORITATIVE_PTID",
            "COHORT_LABEL": "AUTHORITATIVE_LABEL",
            "BASELINE_PHASE": "AUTHORITATIVE_PHASE",
            "BASELINE_DATE": "AUTHORITATIVE_BASELINE_DATE",
        }
    ),
    on="RID",
    how="left",
    validate="one_to_one",
)

mri_authoritative_check["AUTHORITATIVE_BASELINE_DATE"] = pd.to_datetime(
    mri_authoritative_check["AUTHORITATIVE_BASELINE_DATE"],
    errors="coerce",
)

mri_authoritative_check["IN_AUTHORITATIVE_COHORT"] = (
    mri_authoritative_check["AUTHORITATIVE_LABEL"].notna()
)

mri_authoritative_check["PTID_MATCH"] = (
    mri_authoritative_check["PTID"].astype("string")
    == mri_authoritative_check["AUTHORITATIVE_PTID"].astype("string")
)

mri_authoritative_check["LABEL_MATCH"] = (
    mri_authoritative_check["final_group"].astype("string")
    == mri_authoritative_check["AUTHORITATIVE_LABEL"].astype("string")
)

mri_authoritative_check["PHASE_MATCH"] = (
    mri_authoritative_check["baseline_phase"].astype("string")
    == mri_authoritative_check["AUTHORITATIVE_PHASE"].astype("string")
)

mri_authoritative_check["BASELINE_DATE_MATCH"] = (
    mri_authoritative_check["baseline_date"]
    == mri_authoritative_check["AUTHORITATIVE_BASELINE_DATE"]
)

print("\nAlignment with authoritative clinical cohort")
print("--------------------------------------------")
print(
    "MRI participants found in authoritative cohort:",
    f"{mri_authoritative_check['IN_AUTHORITATIVE_COHORT'].sum():,}"
)
print(
    "MRI participants absent from authoritative cohort:",
    f"{(~mri_authoritative_check['IN_AUTHORITATIVE_COHORT']).sum():,}"
)

for column in [
    "PTID_MATCH",
    "LABEL_MATCH",
    "PHASE_MATCH",
    "BASELINE_DATE_MATCH",
]:
    mismatch_count = (
        mri_authoritative_check["IN_AUTHORITATIVE_COHORT"]
        & ~mri_authoritative_check[column].fillna(False)
    ).sum()

    print(f"{column} mismatches: {mismatch_count:,}")

### 42.1. MRI manifest audit interpretation

The MRI manifest itself is structurally sound:

- 1,063 rows and 1,063 unique participants;
- no missing or duplicated identifiers;
- every MRI is within ±90 days of its stored baseline;
- all final RAS, N4, SyN, cropped, normalized T1, and log-Jacobian outputs required for modelling are present;
- all final normalized T1 arrays are valid;
- crop and normalized shapes are consistently `(177, 213, 183)`;
- all available participants agree with the reconstructed cohort on PTID, label, phase, and baseline date.

The failed `mni_npy_exists` and `logjacobian_npy_exists` checks refer to older intermediate array outputs. They do not invalidate the completed pipeline because the later cropped and normalized `.npy` outputs exist for all 1,063 participants.

The only unresolved issue is that six MRI participants are not part of the newly reconstructed 2,199-person authoritative clinical cohort. These six should be inspected before deciding whether the current MRI manifest can be used unchanged or whether a 1,057-participant authoritative-overlap MRI manifest should be created.

In [ ]:
# Inspect the six MRI participants absent from the authoritative cohort

mri_not_in_authoritative = (
    mri_authoritative_check.loc[
        ~mri_authoritative_check["IN_AUTHORITATIVE_COHORT"]
    ]
    .copy()
)

inspection_columns = [
    "RID",
    "PTID",
    "final_group",
    "baseline_diagnosis",
    "baseline_phase",
    "baseline_date",
    "study_date",
    "days_from_baseline_mri",
    "abs_days_from_baseline_mri",
    "research_group",
    "phase",
    "image_id",
    "normalized_t1_npy_path",
    "normalized_t1_npy_valid",
    "cropped_logjacobian_npy_path",
    "cropped_logjacobian_npy_exists",
]

print("MRI participants absent from authoritative cohort")
print("--------------------------------------------------")
print(f"Participants: {len(mri_not_in_authoritative):,}")

display(
    mri_not_in_authoritative[
        inspection_columns
    ].sort_values(
        ["final_group", "RID"]
    )
)

print("\nDistribution of the six excluded MRI participants")
print("--------------------------------------------------")
print(
    mri_not_in_authoritative["final_group"]
    .value_counts(dropna=False)
)

print("\nPresence in the reconstructed MCI trajectory tables")
print("---------------------------------------------------")

excluded_mri_rids = set(
    mri_not_in_authoritative["RID"]
    .dropna()
    .astype(int)
)

### 42.2. Interpretation of the six MRI-only participants

All six unmatched MRI participants were previously labelled `pMCI`, but none appears in the newly reconstructed authoritative four-group cohort.

The blank trajectory-table output does not yet prove why they were excluded. It may mean that the saved trajectory tables were not loaded into the current notebook, or that these RIDs are recorded only in the exclusion file.

The authoritative MCI trajectory outputs will therefore be loaded directly from disk and searched for these six participants before changing the MRI manifest.

In [ ]:
# Load the authoritative MCI trajectory outputs and identify why the six MRI participants are absent

AUTHORITATIVE_COHORT_DIR = (
    MANIFESTS_DIR
    / "authoritative_clinical_cohort"
)

MCI_TRAJECTORY_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_mci_36m_trajectory_labels.csv"
)

MCI_EXCLUSIONS_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "authoritative_mci_trajectory_exclusions.csv"
)

OTHER_DEMENTIA_PATH = (
    AUTHORITATIVE_COHORT_DIR
    / "excluded_baseline_other_dementia.csv"
)

authoritative_mci_trajectory_labels = pd.read_csv(
    MCI_TRAJECTORY_PATH,
    low_memory=False,
)

authoritative_mci_trajectory_exclusions = pd.read_csv(
    MCI_EXCLUSIONS_PATH,
    low_memory=False,
)

excluded_baseline_other_dementia = pd.read_csv(
    OTHER_DEMENTIA_PATH,
    low_memory=False,
)

for table in [
    authoritative_mci_trajectory_labels,
    authoritative_mci_trajectory_exclusions,
    excluded_baseline_other_dementia,
]:
    if "RID" in table.columns:
        table["RID"] = pd.to_numeric(
            table["RID"],
            errors="coerce",
        ).astype("Int64")

excluded_mri_rids = set(
    mri_not_in_authoritative["RID"]
    .dropna()
    .astype(int)
)

print("Search of authoritative MCI outputs")
print("-----------------------------------")
print("MRI-only RIDs:", sorted(excluded_mri_rids))

print("\nFound in retained MCI trajectory labels")
print("---------------------------------------")
retained_matches = authoritative_mci_trajectory_labels.loc[
    authoritative_mci_trajectory_labels["RID"].isin(excluded_mri_rids)
].copy()

print(f"Rows: {len(retained_matches):,}")
display(retained_matches)

print("\nFound in MCI trajectory exclusions")
print("----------------------------------")
exclusion_matches = authoritative_mci_trajectory_exclusions.loc[
    authoritative_mci_trajectory_exclusions["RID"].isin(excluded_mri_rids)
].copy()

print(f"Rows: {len(exclusion_matches):,}")
display(exclusion_matches)

print("\nFound among baseline other-dementia exclusions")
print("------------------------------------------------")
other_dementia_matches = excluded_baseline_other_dementia.loc[
    excluded_baseline_other_dementia["RID"].isin(excluded_mri_rids)
].copy()

print(f"Rows: {len(other_dementia_matches):,}")
display(other_dementia_matches)

# Compact participant-level resolution summary
resolution_rows = []

for rid in sorted(excluded_mri_rids):
    if rid in set(retained_matches["RID"].dropna().astype(int)):
        source = "retained_mci_trajectory"
    elif rid in set(exclusion_matches["RID"].dropna().astype(int)):
        source = "mci_trajectory_exclusion"
    elif rid in set(other_dementia_matches["RID"].dropna().astype(int)):
        source = "baseline_other_dementia"
    else:
        source = "not_found_in_saved_authoritative_outputs"

    resolution_rows.append(
        {
            "RID": rid,
            "AUTHORITATIVE_OUTPUT_LOCATION": source,
        }
    )

resolution_summary = pd.DataFrame(resolution_rows)

print("\nResolution summary")
print("------------------")
display(resolution_summary)

# 43. Create the authoritative MRI overlap manifest

The original MRI manifest will be preserved unchanged as the complete preprocessing record.

A separate authoritative MRI cohort manifest will now be created by restricting it to participants retained in the reconstructed 2,199-person clinical cohort. This removes the six previously labelled pMCI participants whose updated longitudinal trajectories are unstable or revert after Alzheimer’s disease conversion.

In [ ]:
# Create and save the MRI manifest aligned to the authoritative clinical cohort

mri_authoritative_manifest = (
    mri_authoritative_check.loc[
        mri_authoritative_check["IN_AUTHORITATIVE_COHORT"]
    ]
    .copy()
)

# Remove temporary audit columns added during validation
temporary_audit_columns = [
    "RECALCULATED_DAYS_FROM_BASELINE",
    "AUTHORITATIVE_PTID",
    "AUTHORITATIVE_LABEL",
    "AUTHORITATIVE_PHASE",
    "AUTHORITATIVE_BASELINE_DATE",
    "IN_AUTHORITATIVE_COHORT",
    "PTID_MATCH",
    "LABEL_MATCH",
    "PHASE_MATCH",
    "BASELINE_DATE_MATCH",
]

mri_authoritative_manifest = mri_authoritative_manifest.drop(
    columns=[
        column
        for column in temporary_audit_columns
        if column in mri_authoritative_manifest.columns
    ]
)

MRI_AUTHORITATIVE_PATH = (
    MANIFESTS_DIR
    / "baseline_aligned_modalities"
    / "mri"
    / "mri_baseline_aligned_authoritative_cohort_1057.csv"
)

MRI_AUTHORITATIVE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

mri_authoritative_manifest.to_csv(
    MRI_AUTHORITATIVE_PATH,
    index=False,
)

mri_authoritative_reloaded = pd.read_csv(
    MRI_AUTHORITATIVE_PATH,
    low_memory=False,
)

print("Authoritative MRI cohort manifest")
print("---------------------------------")
print("Path:", MRI_AUTHORITATIVE_PATH)
print("Exists:", MRI_AUTHORITATIVE_PATH.exists())
print(f"Rows: {len(mri_authoritative_reloaded):,}")
print(f"Unique RID: {mri_authoritative_reloaded['RID'].nunique():,}")
print(f"Duplicate RID: {mri_authoritative_reloaded['RID'].duplicated().sum():,}")

print("\nCohort distribution")
print("-------------------")
print(
    mri_authoritative_reloaded["final_group"]
    .value_counts()
)

print("\nExpected versus retained")
print("------------------------")
print(f"Original MRI manifest: {len(mri_manifest):,}")
print(f"Excluded under revised trajectory rules: {len(mri_not_in_authoritative):,}")
print(f"Authoritative MRI manifest: {len(mri_authoritative_reloaded):,}")